# E2-G — Binary mask geometry in frozen Cartesian-trained models

Twelve original B1/B2 best checkpoints, two architectures and three seeds are evaluated on the same five matched populations. White marks the original binary iris support; every other pixel is black. All grayscale iris texture is removed. No retraining, label changes, new populations or test-threshold fitting.

Read [REPORT.md](REPORT.md), [METHOD.md](METHOD.md) and [saved results](Results/README.md). All implementations needed by this notebook are visible below; no project-module imports or preceding notebook execution is required. New outputs stay separate from recorded results.

## Imports

All package imports are collected in this first code cell.

In [ ]:
from __future__ import annotations

# Standard library
import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
from collections import Counter, defaultdict
from collections.abc import Mapping, Sequence
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from functools import lru_cache
from importlib import metadata
from numbers import Integral
from pathlib import Path, PureWindowsPath

# Scientific libraries, models and notebook display
import torch
import timm
import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib
matplotlib.use("Agg")  # Same backend as the saved scientific figures.
import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.optimize import least_squares
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import auc, roc_curve
from timm.models.swin_transformer_v2 import WindowAttention
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

# Optional: only explicit native-mask generation needs ONNX Runtime.
_ONNXRUNTIME_IMPORT_ERROR = None
try:
    import onnxruntime as ort
except (ImportError, OSError) as error:
    ort = None
    _ONNXRUNTIME_IMPORT_ERROR = error


## 1. Editable asset paths and execution switches

In [ ]:
REPOSITORY_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents]
    if (p/"Assets/subject_splits.json").is_file()),Path.cwd())
DATA_ROOT=(REPOSITORY_ROOT/"data").resolve()  # EDIT licensed source/cache bundle
EVALUATION_MANIFEST=DATA_ROOT/"evaluation_inventory.parquet"
CHECKPOINT_ROOT=DATA_ROOT/"checkpoints"
CHECKPOINT_INDEX=REPOSITORY_ROOT/"Assets/checkpoints.json"
EMBEDDING_ROOT=DATA_ROOT/"original_embeddings"
INTERVENTION_EMBEDDING_ROOT=DATA_ROOT/"intervention_embeddings"
SUBJECT_SPLITS=REPOSITORY_ROOT/"Assets/subject_splits.json"
MATCHED_POPULATION=REPOSITORY_ROOT/"Assets/Matched Population"
OCCLUSION_POPULATION=REPOSITORY_ROOT/"Assets/Occlusion Population"
ATTRIBUTION_PAIRS=REPOSITORY_ROOT/"Assets/Attribution Pairs"
PATH_REMAP={}
OUTPUT_ROOT=Path("reproduced_outputs/E2-G").resolve()
FROOT=OUTPUT_ROOT
RECORDED_RESULTS=REPOSITORY_ROOT/"E2 Experiments/E2-G/Results"
BASELINE_SOURCE="saved_embeddings"     # or explicitly "recompute" with original full inference order
INTERVENTION_SOURCE="saved_embeddings" # or explicitly "recompute" with frozen best checkpoints
RUN_ANALYSIS=False
SHOW_RECORDED_RESULTS=True
SAVE_LOCAL_BIOMETRIC_PREVIEWS=False
VERIFY_PAIR_SELECTION=False
RUN_NUMERICAL_REPLAY=True
ANALYSIS_CONFIG={'version': 'e2_frozen_scientific_settings', 'datasets': ['CUST-Iris', 'CASIA-Interval', 'IITD', 'AMF', 'UTIRIS_NIR'], 'seeds': [42, 123, 2025], 'normalization': {'mean': 0.48287347719744544, 'std': 0.2741831329802273}, 'metrics': ['eer', 'roc_auc', 'tar_at_far_1pct', 'tar_at_far_0_1pct'], 'sample_sd_ddof': 1, 'quality': {'local_window': 3, 'minimum_interior_samples': 32, 'bright_proxy_uint8_threshold': 250, 'high_gradient_quantile': 0.75, 'radial_bins': 4, 'angular_bins': 8, 'example_quantiles': [0.1, 0.5, 0.9]}, 'perturbation': {'area_fractions': [0.05, 0.1], 'sides': ['left', 'right'], 'regions': ['iris', 'periocular'], 'guard_pixels': 2, 'iris_target_radius_fraction': 0.7, 'periocular_target_radius_fraction': 1.5, 'minimum_disk_radius': 1, 'fill': 'frozen_training_mean', 'alpha': 1.0, 'common_population_requires_all_variants': True}, 'inference': {'batch_size': 24, 'num_workers': 0, 'device': 'cuda', 'amp': 'original_per_checkpoint_policy'}, 'training': False, 'threshold_selection': False, 'domain_pooling': False}
STAGE_CONFIG={'version': 'e2_g_binary_geometry_v1', 'datasets': ['CUST-Iris', 'CASIA-Interval', 'IITD', 'AMF', 'UTIRIS_NIR'], 'architectures': ['ResNet-18', 'SwinV2-Tiny'], 'seeds': [42, 123, 2025], 'checkpoint_conditions': ['E1-B1', 'E1-B2'], 'conditions': ['segmented_grayscale', 'binary_geometry'], 'normalization': {'mean': 0.48287347719744544, 'std': 0.2741831329802273}, 'binary_input': {'foreground_uint8': 255, 'background_uint8': 0, 'shape': [320, 320], 'additional_resize': False, 'soft_edges': False, 'inversion': False}, 'support': {'E1-B1': 'b1_support', 'E1-B2': 'b2_support'}, 'population': 'exact existing E2-A common membership and exhaustive pairs within each dataset', 'inference': {'batch_size': 24, 'num_workers': 0, 'device': 'cuda', 'amp': 'unchanged original per-checkpoint policy'}, 'metrics': ['eer', 'roc_auc', 'tar_at_far_1pct', 'tar_at_far_0_1pct'], 'sample_sd_ddof': 1, 'preview_selection': 'lowest, median, highest predicted lash fraction with distinct subjects; no recognition-score selection', 'subject_sensitivity': 'per-image and per-subject score changes plus leave-one-subject-out metrics', 'training': False, 'threshold_fitting': False, 'domain_pooling': False, 'checkpoint_selection': False}


## 2. Frozen models, image transforms, inventories and numerical definitions

### Scientific settings and environment

In [ ]:
config_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
config_EXTERNAL_DOMAINS = ('AMF', 'UTIRIS_NIR')
config_CONDITIONS = ('E1-A', 'E1-B1', 'E1-B2', 'E1-C1', 'E1-C2')
config_ARCHITECTURES = ('ResNet-18', 'SwinV2-Tiny')
config_SEEDS = (42, 123, 2025)
config_NORMALIZATION = {'mean': 0.48287347719744544, 'std': 0.2741831329802273}

def config_frozen_config(architecture, condition, seed=42, device='cuda'):
    if architecture not in config_ARCHITECTURES or condition not in config_CONDITIONS or seed not in config_SEEDS:
        raise ValueError('Choose the original backbone, E1 condition, and seed 42/123/2025')
    path = None
    config = deepcopy(config_FROZEN_DEFAULTS[architecture][condition])
    config.update(architecture_label=architecture, condition=condition, seed=int(seed))
    config['runtime']['device'] = str(device)
    config['input_shape'] = [1, 64, 512] if condition.startswith('E1-C') else [1, 320, 320]
    config['normalization'] = dict(config_NORMALIZATION)
    return config

def config_environment_info():
    """Display installed versions; do not install packages or change the environment."""
    packages = {}
    for name in ('torch', 'torchvision', 'timm', 'numpy', 'opencv-python', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'nbformat'):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            raise ImportError(f'Missing {name}. Install requirements.txt into the selected notebook kernel environment.') from None
    return {'python': platform.python_version(), **packages, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
config_FROZEN_DEFAULTS = {'ResNet-18': {'E1-A': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1', 'eligibility_policy': 'pure_v2_common_available'}}, 'E1-C2': {'augmentation': {'enabled': True, 'probability': 0.5, 'brightness': 0.1, 'contrast': 0.1, 'horizontal_flip': False, 'random_erasing': False, 'geometric_transforms': False}, 'sampler': {'identities_per_domain': 4, 'captures_per_identity': 2, 'batch_size': 24, 'batches_per_epoch': 224}, 'objective': {'triplet_margin': 0.3, 'supcon_temperature': 0.07, 'supcon_weight': 0.5}, 'optimizer': {'name': 'AdamW', 'lr': 0.0001, 'weight_decay': 0.0001, 'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0}, 'runtime': {'device': 'cuda', 'amp': True, 'amp_dtype': 'float16', 'grad_scaler_init_scale': 1024.0, 'deterministic_algorithms': True, 'num_workers': 0, 'validation_batch_size': 24, 'offline_pretrained': True}, 'schedule': {'max_epochs': 150, 'min_epochs': 30, 'validation_frequency': 1, 'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'selection': 'macro_validation_eer'}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'score_chunk_size': 65536, 'evaluation_augmentation': False, 'evaluate_test': False}, 'logging': {'jsonl': 'history.jsonl', 'curves': 'curves.png', 'write_curves_every_epoch': True}, 'architecture': {'name': 'resnet18.a1_in1k'}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2', 'eligibility_policy': 'pure_v2_common_available'}}}, 'SwinV2-Tiny': {'E1-A': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'raw_ocular'}}, 'E1-B1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'annulus'}}, 'E1-B2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [320, 320]}, 'data': {'normalization_scope': 'pk_eligible_training', 'representation': 'visible_iris'}}, 'E1-C1': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c1'}}, 'E1-C2': {'augmentation': {'brightness': 0.1, 'contrast': 0.1, 'enabled': True, 'geometric_transforms': False, 'horizontal_flip': False, 'probability': 0.5, 'random_erasing': False}, 'sampler': {'batch_size': 24, 'batches_per_epoch': 224, 'captures_per_identity': 2, 'identities_per_domain': 4}, 'objective': {'supcon_temperature': 0.07, 'supcon_weight': 0.5, 'triplet_margin': 0.3}, 'optimizer': {'betas': [0.9, 0.999], 'gradient_clip_norm': 5.0, 'lr': 0.0001, 'name': 'AdamW', 'weight_decay': 0.0001}, 'runtime': {'amp': True, 'amp_dtype': 'float16', 'deterministic_algorithms': True, 'device': 'cuda', 'grad_scaler_init_scale': 1024.0, 'num_workers': 0, 'offline_pretrained': True, 'validation_batch_size': 24}, 'schedule': {'early_stopping_patience': 25, 'improvement_tolerance': 0.0001, 'max_epochs': 150, 'min_epochs': 30, 'selection': 'macro_validation_eer', 'validation_frequency': 1}, 'validation': {'all_genuine_pairs': True, 'all_primary_impostor_pairs': True, 'evaluate_test': False, 'evaluation_augmentation': False, 'score_chunk_size': 65536}, 'logging': {'curves': 'curves.png', 'jsonl': 'history.jsonl', 'write_curves_every_epoch': True}, 'architecture': {'name': 'swinv2_tiny_window8_256', 'pretrained_tag': 'ms_in1k', 'pretrained': True, 'in_chans': 1, 'num_classes': 0, 'global_pool': 'avg', 'strict_img_size': False, 'outer_input_shape_guard': True, 'feature_dim': 768, 'projection_hidden_dim': 512, 'embedding_dim': 256, 'full_fine_tune': True, 'compatibility_decision': 'user_approved_dynamic_attention_masks_all_five_conditions', 'attention_precision': 'float32', 'attention_precision_scope': 'all_12_window_attention_modules', 'remaining_precision': 'original_cuda_amp_float16_with_grad_scaler', 'precision_decision': 'user_approved_float32_window_attention_all_five_conditions', 'input_size': [64, 512]}, 'data': {'eligibility_policy': 'pure_v2_common_available', 'normalization_scope': 'pk_eligible_training', 'representation': 'polar_v2_c2'}}}}

### Photometric augmentation and geometry errors

In [ ]:
class geometry_support_GeometryFailure(ValueError):

    def __init__(self, reason, detail=None):
        self.reason, self.detail = (reason, detail or {})
        super().__init__(reason)

def geometry_support_least_squares_circle(points):
    origin = points.mean(axis=0)
    shifted = points - origin
    matrix = np.column_stack((2 * shifted[:, 0], 2 * shifted[:, 1], np.ones(len(points))))
    solution, _, rank, _ = np.linalg.lstsq(matrix, np.sum(shifted ** 2, axis=1), rcond=None)
    radius2 = solution[2] + solution[0] ** 2 + solution[1] ** 2
    if rank < 3 or radius2 <= 0:
        return None
    return np.array([solution[0] + origin[0], solution[1] + origin[1], math.sqrt(radius2)])

def geometry_support_candidate_circles(points, count, seed):
    rng = np.random.default_rng(seed)
    triples = np.array([rng.choice(len(points), 3, replace=False) for _ in range(count)])
    a, b, c = (points[triples[:, 0]], points[triples[:, 1]], points[triples[:, 2]])
    u, v = (b - a, c - a)
    denominator = 2 * (u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0])
    keep = np.abs(denominator) > 1e-08
    a, u, v, denominator = (a[keep], u[keep], v[keep], denominator[keep])
    uu, vv = ((u * u).sum(1), (v * v).sum(1))
    x = (v[:, 1] * uu - u[:, 1] * vv) / denominator
    y = (u[:, 0] * vv - v[:, 0] * uu) / denominator
    candidates = np.column_stack((a[:, 0] + x, a[:, 1] + y, np.hypot(x, y)))
    algebraic = geometry_support_least_squares_circle(points)
    return np.vstack((algebraic, candidates)) if algebraic is not None else candidates

### Photometric augmentation

In [ ]:
class augmentation_PhotometricAugmentation:
    """Probability0.5: multiplicative brightness and contrast jitter of±10%.

    Brightness is applied first and clamped to[0,1]. Contrast then scales about
    that image's mean, followed by another clamp. Three CPU Torch RNG draws per
    applied transform (gate, brightness, contrast) make checkpoint RNG sufficient.
    No interpolation, geometry, occlusion, erasing or channel changes occur.
    """

    def __init__(self, probability: float=0.5, brightness: float=0.1, contrast: float=0.1):
        if not 0 <= probability <= 1 or not 0 <= brightness <= 1 or (not 0 <= contrast <= 1):
            raise ValueError('Photometric settings must lie in[0,1]')
        self.probability, self.brightness, self.contrast = (probability, brightness, contrast)

    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        if self.probability == 0 or torch.rand(()).item() >= self.probability:
            return image
        brightness = 1 + (2 * torch.rand(()).item() - 1) * self.brightness
        contrast = 1 + (2 * torch.rand(()).item() - 1) * self.contrast
        adjusted = (image * brightness).clamp(0, 1)
        mean = adjusted.mean()
        return ((adjusted - mean) * contrast + mean).clamp(0, 1)

def augmentation_augment_polar(image, validity, probability=0.5, brightness=0.1, contrast=0.1):
    """Photometric jitter uses valid tissue, then explicitly restores invalid zeroes."""
    if image.dtype != torch.float32 or validity.dtype != torch.bool or image.shape != validity.shape:
        raise ValueError('Expected matching float32 texture and Boolean polar validity')
    if not validity.any():
        raise geometry_support_GeometryFailure('no_valid_polar_augmentation_tissue')
    if torch.rand(()).item() < probability:
        bright = 1 + (2 * torch.rand(()).item() - 1) * brightness
        factor = 1 + (2 * torch.rand(()).item() - 1) * contrast
        image = (image * bright).clamp(0, 1)
        mean = image[validity].mean()
        image = ((image - mean) * factor + mean).clamp(0, 1)
    return image.masked_fill(~validity, 0.0)

### Frozen V2 boundary selection, robust fitting and native sampling

In [ ]:
def geometry_v2_configuration():
    return geometry_v2_GEOMETRY_SETTINGS.copy()

def geometry_v2_select_points(mask, role, s):
    if mask.dtype != np.bool_ or mask.ndim != 2:
        raise geometry_support_GeometryFailure(f'{role}_mask_dtype_or_shape')
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype('uint8'), 8)
    if count == 1:
        raise geometry_support_GeometryFailure(f'empty_{role}_mask')
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest = int(areas.argmax()) + 1
    info = {'foreground_pixels': int(mask.sum()), 'components': int(count - 1), 'component_sizes': sorted(areas.tolist(), reverse=True), 'dominant_fraction': float(areas.max() / areas.sum()), 'native_frame_contact': bool(mask[0].any() or mask[-1].any() or mask[:, 0].any() or mask[:, -1].any())}
    if areas.max() < s['minimum_component_pixels']:
        raise geometry_support_GeometryFailure(f'weak_{role}_component', info)
    if info['dominant_fraction'] < s['minimum_dominant_component_fraction']:
        raise geometry_support_GeometryFailure(f'ambiguous_multiple_{role}_components', info)
    contours, _ = cv2.findContours((labels == largest).astype('uint8'), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    full = max(contours, key=cv2.contourArea)[:, 0, :].astype('float64')
    hull = cv2.convexHull(full.astype('float32'))[:, 0, :].astype('float64')
    h, w = mask.shape
    m = s['frame_exclusion_margin_pixels']
    keep = (hull[:, 0] > m) & (hull[:, 0] < w - 1 - m) & (hull[:, 1] > m) & (hull[:, 1] < h - 1 - m)
    points = hull[keep]
    info.update(full_contour_points=len(full), hull_vertices=len(hull), frame_hull_vertices_removed=int((~keep).sum()), selected_points=len(points))
    if len(points) < s['minimum_hull_vertices']:
        raise geometry_support_GeometryFailure(f'insufficient_{role}_curved_boundary', info)
    return (full, points, info)

def geometry_v2_residual(points, model):
    c = np.array([model['center_x'], model['center_y']])
    q = points - c
    if model['kind'] == 'circle':
        return np.linalg.norm(q, axis=1) - model['radius']
    co, si = (np.cos(model['angle_radians']), np.sin(model['angle_radians']))
    u = q[:, 0] * co + q[:, 1] * si
    v = -q[:, 0] * si + q[:, 1] * co
    a, b = (model['axis_a'], model['axis_b'])
    f = (u / a) ** 2 + (v / b) ** 2 - 1
    grad = 2 * np.sqrt((u / a ** 2) ** 2 + (v / b ** 2) ** 2)
    return f / np.maximum(grad, 1e-12)

def geometry_v2_circle_model(c):
    return dict(kind='circle', center_x=float(c[0]), center_y=float(c[1]), radius=float(c[2]), axis_a=float(c[2]), axis_b=float(c[2]), angle_radians=0.0)

def geometry_v2_ellipse_model(c):
    return dict(kind='ellipse', center_x=float(c[0]), center_y=float(c[1]), axis_a=float(c[2]), axis_b=float(c[3]), angle_radians=float(c[4]), radius=float(np.sqrt(c[2] * c[3])))

def geometry_v2_tolerance(model, s):
    return max(s['inlier_absolute_tolerance_pixels'], s['inlier_relative_tolerance'] * model['radius'])

def geometry_v2_circle_fit(points, shape, s):
    maximum = np.hypot(*shape) * s['maximum_radius_fraction_image_diagonal']
    candidates = geometry_support_candidate_circles(points, s['ransac_candidates'], s['deterministic_seed'])
    candidates = candidates[np.isfinite(candidates).all(1) & (candidates[:, 2] >= 1) & (candidates[:, 2] <= maximum)]
    if not len(candidates):
        raise geometry_support_GeometryFailure('no_circle_candidate')
    d = np.linalg.norm(points[None] - candidates[:, None, :2], axis=2) - candidates[:, None, 2]
    tol = np.maximum(s['inlier_absolute_tolerance_pixels'], s['inlier_relative_tolerance'] * candidates[:, 2])
    counts = (np.abs(d) <= tol[:, None]).sum(1)
    score = counts - 0.001 * np.minimum(np.abs(d), 10).sum(1)
    ordered = np.argsort(-score, kind='stable')
    initials = []
    for index in ordered:
        candidate = candidates[index]
        if all((np.linalg.norm(candidate - old) > max(1, 0.025 * candidate[2]) for old in initials)):
            initials.append(candidate)
        if len(initials) >= s['distinct_candidate_refinements']:
            break
    h, w = shape
    fitted = []
    for initial in initials:
        circle = initial.copy()
        for _ in range(s['consensus_refinement_iterations']):
            t = max(s['inlier_absolute_tolerance_pixels'], s['inlier_relative_tolerance'] * circle[2])
            inliers = np.abs(np.linalg.norm(points - circle[:2], axis=1) - circle[2]) <= t
            if inliers.sum() < 6:
                break
            observed = points[inliers]
            fit = least_squares(lambda c: np.linalg.norm(observed - c[:2], axis=1) - c[2], circle, bounds=([-w, -h, 1], [2 * w, 2 * h, maximum]), loss='soft_l1', f_scale=1.0, max_nfev=100, ftol=1e-09, xtol=1e-09, gtol=1e-09)
            circle = fit.x
        model = geometry_v2_circle_model(circle)
        distance = geometry_v2_residual(points, model)
        t = geometry_v2_tolerance(model, s)
        inliers = np.abs(distance) <= t
        outside = float((distance > t).mean())
        eligible = outside <= s['maximum_boundary_outside_fraction'] and geometry_v2_support(points[inliers], model) >= s['minimum_supported_arc_degrees']
        fitted.append((eligible, int(inliers.sum()), -float(np.minimum(np.abs(distance), 10).sum()), model))
    return max(fitted, key=lambda item: item[:3])[3]

def geometry_v2_ellipse_fit(points, shape, s):
    initial = cv2.fitEllipseDirect(points.astype('float32'))
    (cx, cy), (aa, bb), angle = initial
    h, w = shape
    maximum = np.hypot(w, h) * s['maximum_radius_fraction_image_diagonal']
    c = np.array([cx, cy, aa / 2, bb / 2, np.deg2rad(angle)])
    if not np.isfinite(c).all() or min(c[2:4]) < 1 or max(c[2:4]) > maximum:
        raise geometry_support_GeometryFailure('implausible_ellipse_initialization')
    fit = least_squares(lambda p: geometry_v2_residual(points, geometry_v2_ellipse_model(p)), c, bounds=([-w, -h, 1, 1, -np.inf], [2 * w, 2 * h, maximum, maximum, np.inf]), loss='soft_l1', f_scale=1.0, max_nfev=100, ftol=1e-09, xtol=1e-09, gtol=1e-09)
    return geometry_v2_ellipse_model(fit.x)

def geometry_v2_support(points, model):
    theta = np.sort(np.mod(np.arctan2(points[:, 1] - model['center_y'], points[:, 0] - model['center_x']), 2 * np.pi))
    return float(np.degrees(2 * np.pi - np.diff(np.r_[theta, theta[0] + 2 * np.pi]).max())) if len(theta) > 1 else 0.0

def geometry_v2_evaluate(full, points, model, s):
    d = geometry_v2_residual(points, model)
    t = geometry_v2_tolerance(model, s)
    inliers = np.abs(d) <= t
    return dict(inlier_fraction=float(inliers.mean()), supported_arc_degrees=geometry_v2_support(points[inliers], model), full_contour_outside_fraction=float((geometry_v2_residual(full, model) > t).mean()), median_absolute_residual_pixels=float(np.median(np.abs(d))), inlier_median_residual_pixels=float(np.median(np.abs(d[inliers]))) if inliers.any() else None, axis_ratio=float(min(model['axis_a'], model['axis_b']) / max(model['axis_a'], model['axis_b'])))

def geometry_v2_fit_boundary(mask, role, s, diagnostic=False):
    full, points, info = geometry_v2_select_points(mask, role, s)
    model = geometry_v2_circle_fit(points, mask.shape, s)
    circle = model
    circle_qc = geometry_v2_evaluate(full, points, circle, s)
    selection = {'circle': dict(geometry=circle, quality=circle_qc), 'ellipse_considered': False}
    if role in s['ellipse_roles'] and len(points) >= 20:
        try:
            ellipse = geometry_v2_ellipse_fit(points, mask.shape, s)
            eq = geometry_v2_evaluate(full, points, ellipse, s)
            n = len(points)
            rss_circle = max(float((geometry_v2_residual(points, circle) ** 2).sum()), 1e-12)
            rss_ellipse = max(float((geometry_v2_residual(points, ellipse) ** 2).sum()), 1e-12)
            bic_improvement = n * np.log(rss_circle / rss_ellipse) - 2 * np.log(n)
            hold_circle = []
            hold_ellipse = []
            for fold in range(s['holdout_folds']):
                test = np.arange(n) % s['holdout_folds'] == fold
                cm = geometry_v2_circle_fit(points[~test], mask.shape, s)
                em = geometry_v2_ellipse_fit(points[~test], mask.shape, s)
                hold_circle.extend(np.abs(geometry_v2_residual(points[test], cm)))
                hold_ellipse.extend(np.abs(geometry_v2_residual(points[test], em)))
            error_ratio = float(np.mean(hold_ellipse) / max(np.mean(hold_circle), 1e-12))
            eligible = eq['axis_ratio'] >= s['ellipse_minimum_axis_ratio'] and eq['supported_arc_degrees'] >= s['ellipse_minimum_supported_arc_degrees'] and (eq['inlier_fraction'] >= s[f'minimum_{role}_inlier_fraction']) and (eq['full_contour_outside_fraction'] <= s['maximum_boundary_outside_fraction']) and (bic_improvement >= s['ellipse_required_bic_improvement']) and (error_ratio <= s['ellipse_required_holdout_error_ratio'])
            selection.update(ellipse_considered=True, ellipse=dict(geometry=ellipse, quality=eq), bic_improvement=float(bic_improvement), holdout_error_ratio=error_ratio, ellipse_eligible=bool(eligible))
            if eligible:
                model = ellipse
        except (geometry_support_GeometryFailure, cv2.error, ValueError) as error:
            selection['ellipse_fit_failure'] = str(error)
    qc = geometry_v2_evaluate(full, points, model, s)
    detail = {**info, **model, **qc, 'model_selection': selection}
    reason = None
    if qc['axis_ratio'] < s['ellipse_minimum_axis_ratio']:
        reason = f'implausible_{role}_axis_ratio'
    elif qc['inlier_fraction'] < s[f'minimum_{role}_inlier_fraction']:
        reason = f'weak_{role}_curved_support'
    elif qc['supported_arc_degrees'] < s['minimum_supported_arc_degrees']:
        reason = f'insufficient_{role}_angular_support'
    elif qc['full_contour_outside_fraction'] > s['maximum_boundary_outside_fraction']:
        reason = f'{role}_boundary_outside_fit'
    if reason:
        raise geometry_support_GeometryFailure(reason, detail)
    return detail

def geometry_v2_boundary(model, theta):
    vector = np.stack((np.cos(theta), np.sin(theta)), axis=-1)
    angle = theta - model['angle_radians']
    a, b = (model['axis_a'], model['axis_b'])
    radius = 1 / np.sqrt((np.cos(angle) / a) ** 2 + (np.sin(angle) / b) ** 2)
    return np.array([model['center_x'], model['center_y']]) + radius[:, None] * vector

def geometry_v2_validate_pair(pupil, iris, s):
    n = s['containment_directions']
    theta = np.arange(n) * 2 * np.pi / n
    vector = np.stack((np.cos(theta), np.sin(theta)), axis=1)

    def h(model):
        t = theta - model['angle_radians']
        return np.sqrt((model['axis_a'] * np.cos(t)) ** 2 + (model['axis_b'] * np.sin(t)) ** 2)
    delta = np.array([iris['center_x'] - pupil['center_x'], iris['center_y'] - pupil['center_y']])
    margins = vector @ delta + h(iris) - h(pupil)
    lipschitz = np.linalg.norm(delta) + max(iris['axis_a'], iris['axis_b']) + max(pupil['axis_a'], pupil['axis_b'])
    certified_margin = float(margins.min() - lipschitz * np.pi / n)
    ratio = pupil['radius'] / iris['radius']
    info = dict(center_offset_pixels=float(np.linalg.norm(delta)), pupil_to_iris_radius_ratio=float(ratio), certified_containment_margin_pixels=certified_margin, containment_directions=n)
    if not s['minimum_pupil_to_iris_radius_ratio'] <= ratio <= s['maximum_pupil_to_iris_radius_ratio']:
        raise geometry_support_GeometryFailure('pupil_iris_radius_ratio', info)
    if certified_margin <= 0:
        raise geometry_support_GeometryFailure('boundaries_not_certifiably_nested', info)
    inner, outer = (geometry_v2_boundary(pupil, theta), geometry_v2_boundary(iris, theta))
    dr = outer - inner
    v = vector
    info['minimum_radial_progress_pixels'] = float((dr * v).sum(1).min())
    derivative = []
    for model in (pupil, iris):
        t = theta - model['angle_radians']
        a, b = (model['axis_a'], model['axis_b'])
        rho = 1 / np.sqrt((np.cos(t) / a) ** 2 + (np.sin(t) / b) ** 2)
        prime = rho ** 3 * np.sin(t) * np.cos(t) * (1 / a ** 2 - 1 / b ** 2)
        derivative.append(prime[:, None] * v + rho[:, None] * np.stack((-np.sin(theta), np.cos(theta)), 1))
    jac = [dr[:, 0] * d[:, 1] - dr[:, 1] * d[:, 0] for d in derivative]
    info['minimum_mapping_jacobian'] = float(min((j.min() for j in jac)))
    if info['minimum_radial_progress_pixels'] <= 0 or info['minimum_mapping_jacobian'] <= 0:
        raise geometry_support_GeometryFailure('non_monotone_mapping', info)
    return info

def geometry_v2_coordinates(pupil, iris, height=64, width=512):
    theta = np.arange(width) * 2 * np.pi / width
    r = (np.arange(height) + 0.5) / height
    inner, outer = (geometry_v2_boundary(pupil, theta), geometry_v2_boundary(iris, theta))
    p = (1 - r[:, None, None]) * inner[None] + r[:, None, None] * outer[None]
    return (p[:, :, 0], p[:, :, 1])

def geometry_v2_sample_coordinates(gray, annulus, lashes, x, y):
    if gray.dtype != np.uint8 or gray.ndim != 2 or gray.shape != annulus.shape or (gray.shape != lashes.shape):
        raise geometry_support_GeometryFailure('native_alignment')
    h, w = gray.shape
    inframe = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    xn = np.clip(np.floor(x + 0.5).astype(int), 0, w - 1)
    yn = np.clip(np.floor(y + 0.5).astype(int), 0, h - 1)
    nn = annulus[yn, xn] & inframe
    occlusion = lashes[yn, xn] & inframe
    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)
    dx = x - x0
    dy = y - y0
    value = np.zeros_like(x)
    footprint = np.ones_like(inframe)
    lash_footprint = np.zeros_like(inframe)
    for xx, yy, weight in ((x0, y0, (1 - dx) * (1 - dy)), (x0 + 1, y0, dx * (1 - dy)), (x0, y0 + 1, (1 - dx) * dy), (x0 + 1, y0 + 1, dx * dy)):
        active = weight > 1e-12
        inside = (xx >= 0) & (xx < w) & (yy >= 0) & (yy < h)
        xc, yc = (np.clip(xx, 0, w - 1), np.clip(yy, 0, h - 1))
        value += weight * gray[yc, xc]
        footprint &= ~active | inside & annulus[yc, xc]
        lash_footprint |= active & inside & lashes[yc, xc]
    valid = nn & footprint
    if not valid.any():
        raise geometry_support_GeometryFailure('no_valid_samples')
    strip = np.rint(value).clip(0, 255).astype('uint8')
    strip[~valid] = 0
    seam = valid[:, 0] & valid[:, -1]
    quality = dict(samples=int(strip.size), valid_samples=int(valid.sum()), valid_fraction=float(valid.mean()), invalid_samples=int((~valid).sum()), out_of_frame_samples=int((~inframe).sum()), interpolation_footprint_rejected_samples=int((nn & ~footprint).sum()), eyelash_samples_retained=int((valid & occlusion).sum()), valid_zero_samples=int(((strip == 0) & valid).sum()), seam_valid_rows=int(seam.sum()), seam_mean_absolute_delta=float(np.abs(strip[:, 0].astype(float) - strip[:, -1])[seam].mean()) if seam.any() else None)
    return (strip, valid, occlusion & valid, lash_footprint & valid, quality)

def geometry_v2_normalize(gray, masks, config):
    geometry = {}
    try:
        for role in ('iris', 'pupil'):
            if not masks[role].any():
                raise geometry_support_GeometryFailure(f'empty_{role}_mask', {'foreground_pixels': 0})
        for role in ('iris', 'pupil'):
            geometry[role] = geometry_v2_fit_boundary(masks[role], role, config['geometry'])
        geometry['pair'] = geometry_v2_validate_pair(geometry['pupil'], geometry['iris'], config['geometry'])
        x, y = geometry_v2_coordinates(geometry['pupil'], geometry['iris'])
        strip, valid, occlusion, lash_footprint, qc = geometry_v2_sample_coordinates(gray, masks['annulus'], masks['eyelashes'], x, y)
        geometry['mapping_quality'] = qc
        return (strip, valid, occlusion, lash_footprint, x, y, geometry)
    except geometry_support_GeometryFailure as error:
        raise geometry_support_GeometryFailure(error.reason, dict(completed_fits=geometry, failure_detail=error.detail)) from error
geometry_v2_GEOMETRY_SETTINGS = {'version': 2, 'experiment': 'E1-C', 'output_height': 64, 'output_width': 512, 'angular_origin': 'positive_x_3_oclock', 'angular_direction': 'clockwise_in_native_image_coordinates', 'radial_positions': 'cell_midpoints', 'required_splits': ['train', 'validation'], 'geometry': {'contour': 'external_dominant_component_convex_hull_vertices', 'frame_exclusion_margin_pixels': 1, 'minimum_component_pixels': 16, 'minimum_dominant_component_fraction': 0.9, 'ambiguous_pupil_components': 'unique_supported_nested_component_only', 'minimum_hull_vertices': 12, 'ransac_candidates': 384, 'consensus_refinement': 'bounded_inlier_only_two_iterations', 'consensus_refinement_iterations': 2, 'distinct_candidate_refinements': 3, 'deterministic_seed': 0, 'inlier_absolute_tolerance_pixels': 1.0, 'inlier_relative_tolerance': 0.025, 'minimum_iris_inlier_fraction': 0.45, 'minimum_pupil_inlier_fraction': 0.6, 'minimum_supported_arc_degrees': 180.0, 'maximum_boundary_outside_fraction': 0.15, 'maximum_radius_fraction_image_diagonal': 0.5, 'ellipse_roles': ['pupil'], 'ellipse_minimum_axis_ratio': 0.5, 'ellipse_minimum_supported_arc_degrees': 300.0, 'ellipse_required_bic_improvement': 10.0, 'ellipse_required_holdout_error_ratio': 0.7, 'holdout_folds': 4, 'containment_directions': 4096, 'minimum_pupil_to_iris_radius_ratio': 0.05, 'maximum_pupil_to_iris_radius_ratio': 0.95, 'fallback': 'none'}, 'sampling': {'native_unmasked_grayscale': True, 'texture': 'bilinear', 'validity': 'nearest_original_annulus_and_active_bilinear_footprint', 'eyelashes': 'retained_with_nearest_audit_mask', 'invalid_value': 0, 'coordinates': 'saved_float64_npz', 'enhancement': 'none', 'inpainting': False}, 'normalization': {'mean': 0.48287347719744544, 'std': 0.2741831329802273}, 'test_pixel_access': False}

### Unique supported pupil-component resolution

In [ ]:
def components_v2_normalize(gray, masks, config):
    try:
        return geometry_v2_normalize(gray, masks, config)
    except geometry_support_GeometryFailure as original:
        if original.reason != 'ambiguous_multiple_pupil_components':
            raise
        if config['geometry']['ambiguous_pupil_components'] != 'unique_supported_nested_component_only':
            raise
        iris = original.detail['completed_fits']['iris']
        count, labels, stats, _ = cv2.connectedComponentsWithStats(masks['pupil'].astype('uint8'), 8)
        candidates = []
        accepted = []
        for label in range(1, count):
            entry = dict(component_label=label, pixels=int(stats[label, cv2.CC_STAT_AREA]))
            try:
                model = geometry_v2_fit_boundary(labels == label, 'pupil', config['geometry'])
                pair = geometry_v2_validate_pair(model, iris, config['geometry'])
                entry.update(status='plausible', geometry=model, pair=pair)
                accepted.append(label)
            except geometry_support_GeometryFailure as error:
                entry.update(status='rejected', reason=error.reason, detail=error.detail)
            candidates.append(entry)
        resolution = dict(method='unique_geometrically_supported_nested_pupil_component', candidates=candidates, plausible_components=len(accepted), original_annulus_unchanged=True)
        if len(accepted) != 1:
            raise geometry_support_GeometryFailure(original.reason, {**original.detail, 'component_resolution': resolution}) from original
        selected = accepted[0]
        fit_masks = dict(masks, pupil=labels == selected)
        result = list(geometry_v2_normalize(gray, fit_masks, config))
        resolution['selected_component_label'] = selected
        result[-1]['pupil_component_resolution'] = resolution
        result[-1]['original_pupil_foreground_pixels'] = int(masks['pupil'].sum())
        return tuple(result)

### Native semantic masks: explicit Worldcoin inference, never silently regenerated

In [ ]:
segmentation_CLASSES = ('eyeball', 'iris', 'pupil', 'eyelashes', 'annulus', 'visible_iris')
segmentation_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
segmentation_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def segmentation_preprocess(gray: np.ndarray, height: int, width: int) -> np.ndarray:
    resized = cv2.resize(gray, (width, height), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB).astype(np.float32) / np.float32(255.0)
    return np.ascontiguousarray(((rgb - segmentation_MEAN) / segmentation_STD).transpose(2, 0, 1))

def segmentation_probability_masks(prediction: np.ndarray, height: int, width: int, threshold: float) -> dict[str, np.ndarray]:
    if prediction.ndim != 3 or prediction.shape[0] != 4:
        raise ValueError(f'Unexpected output shape: {prediction.shape}')
    if not np.isfinite(prediction).all() or prediction.min() < -1e-05 or prediction.max() > 1.00001:
        raise ValueError('Expected finite probabilities in [0, 1]; refusing to threshold logits')
    probabilities = cv2.resize(prediction.transpose(1, 2, 0), (width, height), interpolation=cv2.INTER_LINEAR)
    masks = {name: probabilities[:, :, index] > threshold for index, name in enumerate(segmentation_CLASSES[:4])}
    masks['annulus'] = masks['iris'] & ~masks['pupil']
    masks['visible_iris'] = masks['annulus'] & ~masks['eyelashes']
    return masks

def segmentation_make_session(model: Path, device: str, threads: int):
    if ort is None:
        raise ImportError(
            "Native-mask generation needs the optional ONNX Runtime dependency documented in ASSETS.md."
        ) from _ONNXRUNTIME_IMPORT_ERROR
    available = ort.get_available_providers()
    if device == 'cuda' and 'CUDAExecutionProvider' not in available:
        raise RuntimeError(f'CUDA requested but unavailable: {available}')
    providers = ['CPUExecutionProvider']
    if device != 'cpu' and 'CUDAExecutionProvider' in available:
        providers.insert(0, 'CUDAExecutionProvider')
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(str(model), sess_options=options, providers=providers)
    if device == 'cuda' and 'CUDAExecutionProvider' not in session.get_providers():
        raise RuntimeError('CUDA initialization failed; refusing an unexpected CPU fallback')
    inputs = session.get_inputs()
    if len(inputs) != 1 or inputs[0].type != 'tensor(float)':
        raise ValueError(f'Unsupported model inputs: {inputs}')
    shape = inputs[0].shape
    if len(shape) != 4 or shape[1] != 3 or (not all((isinstance(x, int) and x > 0 for x in shape[2:]))):
        raise ValueError(f'Expected NCHW input with fixed spatial dimensions: {shape}')
    print(f'ONNX Runtime {ort.__version__}; providers={session.get_providers()}; input={shape}', flush=True)
    return (session, inputs[0], ort.__version__)

def segmentation_segment_native(gray, session, model_input):
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise ValueError('Expected native uint8 grayscale')
    x = segmentation_preprocess(gray, *model_input.shape[2:])[None]
    prediction = session.run(None, {model_input.name: x})[0]
    if prediction.shape[0] != 1:
        raise ValueError('Expected one image prediction')
    return segmentation_probability_masks(prediction[0], *gray.shape, threshold=0.5)

### Image, mask and tensor transformations

In [ ]:
def preprocessing_read_gray(path, *, native=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing image: {path}. Configure DATA_ROOT/PATH_REMAP or supply the original private cached representation.')
    flags = cv2.IMREAD_GRAYSCALE | cv2.IMREAD_IGNORE_ORIENTATION if native else cv2.IMREAD_UNCHANGED
    image = cv2.imdecode(np.fromfile(path, np.uint8), flags)
    if image is None or image.dtype != np.uint8 or image.ndim != 2:
        raise ValueError(f'Expected unchanged uint8 grayscale pixels: {path}')
    return image

def preprocessing_read_binary(path, shape=None):
    mask = preprocessing_read_gray(path)
    if shape is not None and mask.shape != tuple(shape):
        raise ValueError(f'Image/mask geometry mismatch: {path}')
    if not np.isin(mask, [0, 255]).all():
        raise ValueError(f'Expected an original 0/255 binary mask; no threshold repair: {path}')
    return mask == 255

def preprocessing_geometry(width, height):
    scale = min(320 / width, 320 / height)
    rw, rh = (round(width * scale), round(height * scale))
    left, top = ((320 - rw) // 2, (320 - rh) // 2)
    return {'native_width': width, 'native_height': height, 'content_width': rw, 'content_height': rh, 'pad_left': left, 'pad_top': top, 'pad_right': 320 - rw - left, 'pad_bottom': 320 - rh - top, 'output_width': 320, 'output_height': 320}

def preprocessing_resize_raw(source, plan):
    dimensions = (plan['content_width'], plan['content_height'])
    content = source.copy() if source.shape == (dimensions[1], dimensions[0]) else cv2.resize(source, dimensions, interpolation=cv2.INTER_AREA)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0)

def preprocessing_transform_support(mask, plan):
    if mask.dtype != np.bool_ or mask.shape != (plan['native_height'], plan['native_width']):
        raise ValueError('Native mask and raw-image geometry must match')
    dimensions = (plan['content_width'], plan['content_height'])
    content = mask.astype(np.uint8)
    if content.shape != (dimensions[1], dimensions[0]):
        content = cv2.resize(content, dimensions, interpolation=cv2.INTER_NEAREST_EXACT)
    return cv2.copyMakeBorder(content, plan['pad_top'], plan['pad_bottom'], plan['pad_left'], plan['pad_right'], cv2.BORDER_CONSTANT, value=0).astype(bool)

def preprocessing_native_cartesian(gray, masks=None):
    """Exact original letterbox and masks. Never stretch or crop around the iris."""
    plan = preprocessing_geometry(gray.shape[1], gray.shape[0])
    raw = preprocessing_resize_raw(gray, plan)
    if masks is None:
        return (raw, None, None)
    annulus = masks['iris'] & ~masks['pupil']
    return (raw, preprocessing_transform_support(annulus, plan), preprocessing_transform_support(annulus & ~masks['eyelashes'], plan))

def preprocessing_native_polar_v2(gray, masks):
    """Frozen geometry, including unique supported pupil-component resolution.

    Rebuilding is explicit. Geometry failures propagate, never become blank
    strips or V1 fallbacks. C1/C2 share exactly the same coordinates and samples.
    """
    if 'annulus' not in masks:
        masks = dict(masks, annulus=masks['iris'] & ~masks['pupil'])
    strip, valid, occlusion, footprint, x, y, fitted = components_v2_normalize(gray, masks, geometry_v2_configuration())
    clean_valid = valid & ~footprint
    if not clean_valid.any():
        raise ValueError('no_valid_non_lash_tissue: keep this image unavailable to both C1/C2')
    clean = strip.copy()
    clean[~clean_valid] = 0
    return {'c1': strip, 'c1_validity': valid, 'c2': clean, 'c2_validity': clean_valid, 'x': x, 'y': y, 'geometry': fitted, 'eyelash_footprint': footprint, 'eyelash_nearest': occlusion}

def preprocessing_tensor_from_pixels(pixels, condition, normalization, *, support=None, validity=None, augmentation=None):
    image = torch.from_numpy(np.ascontiguousarray(pixels[None])).float().div_(255.0)
    if condition.startswith('E1-C'):
        if validity is None or validity.shape != pixels.shape or (not validity.any()) or pixels[~validity].any():
            raise ValueError('Polar pixels require matching nonempty validity and zero invalid pixels')
        if augmentation is not None:
            image = augmentation_augment_polar(image, torch.from_numpy(validity[None]), **augmentation)
    else:
        if augmentation is not None:
            image = augmentation_PhotometricAugmentation(**augmentation)(image)
        if condition in ('E1-B1', 'E1-B2'):
            if support is None or support.shape != pixels.shape or support.dtype != np.bool_:
                raise ValueError('Cartesian segmented inputs require the corresponding binary support')
            image = image.masked_fill(~torch.from_numpy(support)[None], 0.0)
    image = (image - normalization['mean']) / normalization['std']
    shape = (1, 64, 512) if condition.startswith('E1-C') else (1, 320, 320)
    if image.shape != shape or image.dtype != torch.float32 or (not torch.isfinite(image).all()):
        raise ValueError(f'Expected finite float32 input {shape}, got {tuple(image.shape)}')
    return image

### Frozen inventory, physical-eye identities and representation availability

In [ ]:
data_PATH_COLUMNS = ('native_path', 'raw_path', 'b1_support_path', 'b2_support_path', 'c1_path', 'c1_validity_path', 'c2_path', 'c2_validity_path', 'iris_mask_path', 'pupil_mask_path', 'eyelashes_mask_path')

def data_load_table(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing frozen metadata/pair table: {path}. Supply the documented CSV/Parquet asset; this workflow never invents an inventory or resamples a split.')
    if path.suffix.lower() == '.parquet':
        frame = pd.read_parquet(path)
    elif path.suffix.lower() == '.csv':
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    else:
        raise ValueError(f'Expected CSV or Parquet, got {path}')
    return frame

def data_explicit_bool(value, name):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value in ('true', 'True', '1', 1):
        return True
    if value in ('false', 'False', '0', 0):
        return False
    raise ValueError(f'{name} must be an explicit boolean, got {value!r}')

def data_present(value):
    return value is not None and (not (isinstance(value, float) and np.isnan(value))) and (str(value) != '')

def data_resolve_path(value, data_root, path_remap=None):
    if not data_present(value):
        return None
    text = str(value).replace('\\', '/')
    for old, new in sorted((path_remap or {}).items(), key=lambda item: -len(str(item[0]))):
        prefix = str(old).replace('\\', '/').rstrip('/')
        if text == prefix or text.startswith(prefix + '/'):
            return (Path(new).expanduser() / text[len(prefix):].lstrip('/')).resolve()
    path = Path(text).expanduser()
    if PureWindowsPath(text).is_absolute() and (not path.is_absolute()):
        raise ValueError(f'Foreign absolute path {text!r}. Set PATH_REMAP explicitly; no old user directory is assumed.')
    return (path if path.is_absolute() else Path(data_root) / path).resolve()

def data_load_inventory(manifest, *, data_root, subject_splits, image_exclusions, path_remap=None, strict_original_counts=True):
    """Validate frozen identity/split semantics, never file hashes or source seals."""
    frame = data_load_table(manifest)
    required = {'image_id', 'dataset', 'subject_id', 'eye_side', 'iris_id', 'source_id', 'split'}
    if (missing := (required - set(frame))):
        raise ValueError(f'Image manifest lacks {sorted(missing)}. See ASSETS.md for the portable schema; source_id must be the original canonical-content identifier.')
    frozen = json.loads(Path(subject_splits).read_text(encoding='utf-8'))
    exclusions = json.loads(Path(image_exclusions).read_text(encoding='utf-8'))
    if frozen['seed'] != 42 or set(frozen['datasets']) != set(config_DOMAINS):
        raise ValueError('Supply the original seed-42 subject allocations for the three primary datasets')
    if strict_original_counts:
        canonical = data_FROZEN_MEMBERSHIP

        def allocations(value):
            return {d: {s: sorted(map(str, ids)) for s, ids in splits.items()} for d, splits in value.items()}
        if allocations(frozen['datasets']) != allocations(canonical['datasets']):
            raise ValueError('Supplied labels change the original seed-42 subject allocations; no replacement split is permitted')
        if {r['image_id'] for r in exclusions['excluded_images']} != set(canonical['excluded_image_ids']) or {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']} != {(r['dataset'], r['subject_id']) for r in canonical['excluded_subjects']}:
            raise ValueError('Supplied image/subject exclusions differ from the original approved exclusions')
    assignment = {}
    for domain, partitions in frozen['datasets'].items():
        for split, subjects in partitions.items():
            for subject in subjects:
                key = (domain, str(subject))
                if key in assignment:
                    raise ValueError(f'Frozen subject occurs in multiple partitions: {key}')
                assignment[key] = split
    excluded_ids = {r['image_id'] for r in exclusions['excluded_images']}
    excluded_subjects = {(r['dataset'], r['subject_id']) for r in exclusions['excluded_subjects']}
    rows, excluded = ([], [])
    seen = set()
    for row in frame.to_dict('records'):
        for key in required:
            if not data_present(row[key]):
                raise ValueError(f'Missing {key}; labels/source identity may not be guessed')
            row[key] = str(row[key])
        if row['image_id'] in seen:
            raise ValueError(f"Duplicate image ID: {row['image_id']}")
        seen.add(row['image_id'])
        if row['eye_side'] not in ('L', 'R') or row['iris_id'] != f"{row['dataset']}:{row['subject_id']}:{row['eye_side']}":
            raise ValueError(f"Identity/eye namespace mismatch: {row['image_id']}")
        key = (row['dataset'], row['subject_id'])
        if row['image_id'] in excluded_ids or key in excluded_subjects:
            excluded.append(row['image_id'])
            continue
        if row['dataset'] in config_DOMAINS:
            if key not in assignment or row['split'] != assignment[key]:
                raise ValueError(f'Manifest changes the frozen dataset+subject partition: {key}')
        elif row['dataset'] in config_EXTERNAL_DOMAINS:
            if row['split'] != 'external':
                raise ValueError('AMF and UTIRIS_NIR must remain external, never training/validation')
        else:
            raise ValueError(f"Dataset outside E1 scope: {row['dataset']}")
        if 'included' in row and data_present(row['included']) and (not data_explicit_bool(row['included'], 'included')):
            raise ValueError(f"Unapproved additional exclusion: {row['image_id']}. Supply the retained frozen inventory, including QC flags.")
        for column in data_PATH_COLUMNS:
            row[column] = data_resolve_path(row.get(column), data_root, path_remap)
        row['quality_flags'] = row.get('quality_flags', [])
        rows.append(row)
    if not rows:
        raise ValueError('No retained images in the supplied frozen inventory')
    sources = defaultdict(list)
    for row in rows:
        sources[row['source_id']].append(row)
    duplicates = {source: group for source, group in sources.items() if len(group) > 1}
    if duplicates:
        raise ValueError('Redundant canonical source images remain in the included inventory; use the existing audited exact-duplicate exclusions, not a new deduplication rule')
    if strict_original_counts:
        expected = data_INVENTORY_COUNTS
        actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS))
        required_counts = {(d, s): n for d, splits in expected['raw'].items() for s, n in splits.items()}
        if actual != required_counts:
            raise ValueError('Portable manifest is not the complete frozen primary inventory. Supply the original author metadata (all retained train/validation/test rows); do not invent a subset or replacement split. Synthetic software tests must explicitly disable this count check.')
        if 'c1_status' in frame and 'c2_status' in frame:
            actual = Counter(((r['dataset'], r['split']) for r in rows if r['dataset'] in config_DOMAINS and r['c1_status'] == r['c2_status'] == 'ok'))
            required_counts = {(d, s): n for d, splits in expected['polar_common'].items() for s, n in splits.items()}
            if actual != required_counts:
                raise ValueError('Pure-V2 common availability differs from the original audit; supply the frozen status/eligibility manifest rather than new exclusions')
    return (sorted(rows, key=lambda r: r['image_id']), {'images': len(rows), 'approved_exclusion_rows_filtered': excluded, 'groups': dict(Counter(((r['dataset'], r['split']) for r in rows))), 'new_subject_assignments': False, 'file_hash_checks': False})

def data_condition_view(records, condition, split):
    """Keep the historical common C1/C2 availability; report every failure."""
    if split not in ('train', 'validation', 'test', 'external'):
        raise ValueError('Unknown split')
    selected, unavailable = ([], [])
    for row in records:
        if row['split'] != split:
            continue
        if condition in ('E1-B1', 'E1-B2'):
            key = 'b1_status' if condition == 'E1-B1' else 'b2_status'
            if key in row and row[key] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row[key]})
                continue
        if condition.startswith('E1-C'):
            if not all((name in row for name in ('c1_status', 'c2_status', 'geometry_version', 'v1_fallback_used', 'v3_used'))):
                raise ValueError('Polar reproduction requires the frozen pure-V2 availability/status columns and C1/C2 assets; supply the portable V2 manifest, not guessed geometry')
            if str(row['geometry_version']) not in ('v2', '2') or data_explicit_bool(row['v1_fallback_used'], 'v1_fallback_used') or data_explicit_bool(row['v3_used'], 'v3_used'):
                raise ValueError('Only pure V2 is allowed; no V1/hybrid/V3 substitutions')
            if row['c1_status'] != 'ok' or row['c2_status'] != 'ok':
                unavailable.append({'image_id': row['image_id'], 'reason': row.get('failure_reason', 'V2 common availability')})
                continue
            key = 'c1' if condition == 'E1-C1' else 'c2'
            if row.get(key + '_path') is None or row.get(key + '_validity_path') is None:
                raise FileNotFoundError(f"Missing frozen {key.upper()} texture/validity for {row['image_id']}. Supply the original cache or explicitly rebuild using the frozen V2 method and audited availability.")
        selected.append(dict(row))
    if not selected:
        raise ValueError(f'No eligible {split} images for {condition}')
    return (selected, unavailable)

def data_pk_eligible(records):
    sources = defaultdict(set)
    for row in records:
        if row['split'] != 'train':
            raise ValueError('PK eligibility must use training records only')
        sources[row['iris_id']].add(row['source_id'])
    return [r for r in records if len(sources[r['iris_id']]) >= 2]

class data_IrisDataset(Dataset):
    """Exact cached/native tensors with explicit split access and no resizing of caches."""

    def __init__(self, records, config, *, split, augment=False):
        self.records = list(records)
        self.condition = config['condition']
        self.normalization = dict(config['normalization'])
        self.evaluation = not augment
        if not self.records or any((r['split'] != split for r in self.records)):
            raise ValueError('Dataset access requires one explicit, matching partition')
        if augment and split != 'train':
            raise ValueError('Augmentation is permitted on training captures only')
        aug = config['augmentation']
        self.augmentation = {k: aug[k] for k in ('probability', 'brightness', 'contrast')} if augment and aug['enabled'] else None
        self.dataset_labels = {d: i for i, d in enumerate((*config_DOMAINS, *config_EXTERNAL_DOMAINS))}
        self.subject_labels = {k: i for i, k in enumerate(sorted({(r['dataset'], r['subject_id']) for r in self.records}))}
        self.iris_labels = {k: i for i, k in enumerate(sorted({r['iris_id'] for r in self.records}))}
        self.source_labels = {k: i for i, k in enumerate(sorted({r['source_id'] for r in self.records}))}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        support = validity = None
        native_shape = (int(row['height']), int(row['width'])) if data_present(row.get('height')) and data_present(row.get('width')) else None
        if self.condition.startswith('E1-C'):
            key = 'c1' if self.condition == 'E1-C1' else 'c2'
            pixels = preprocessing_read_gray(row[key + '_path'])
            validity = preprocessing_read_binary(row[key + '_validity_path'], (64, 512))
        else:
            if row.get('raw_path') is not None:
                pixels = preprocessing_read_gray(row['raw_path'])
                if pixels.shape != (320, 320):
                    raise ValueError('Cached raw pixels must already be 320x320; no extra resize is allowed')
            else:
                if row.get('native_path') is None:
                    raise FileNotFoundError('Provide raw_path or native_path in the image manifest')
                native = preprocessing_read_gray(row['native_path'], native=True)
                if native_shape is not None and native.shape != native_shape:
                    raise ValueError('Native image dimensions disagree with the authoritative inventory')
                native_shape = native.shape
                pixels = preprocessing_resize_raw(native, preprocessing_geometry(native.shape[1], native.shape[0]))
            if self.condition in ('E1-B1', 'E1-B2'):
                column = 'b1_support_path' if self.condition == 'E1-B1' else 'b2_support_path'
                if row.get(column) is not None:
                    support = preprocessing_read_binary(row[column], (320, 320))
                else:
                    if any((row.get(k + '_mask_path') is None for k in ('iris', 'pupil', 'eyelashes'))):
                        raise FileNotFoundError('Provide the frozen support cache or the native iris/pupil/eyelash masks; no segmentation model is substituted')
                    masks = {k: preprocessing_read_binary(row[k + '_mask_path']) for k in ('iris', 'pupil', 'eyelashes')}
                    if len({m.shape for m in masks.values()}) != 1:
                        raise ValueError('Native segmentation masks do not align')
                    if native_shape is None:
                        if row.get('native_path') is None:
                            raise ValueError('Native-mask fallback needs original width/height or native_path to verify alignment with the raw frame')
                        native_shape = preprocessing_read_gray(row['native_path'], native=True).shape
                    if masks['iris'].shape != native_shape:
                        raise ValueError('Native mask/image geometry mismatch; do not resize mismatched anatomy into apparent alignment')
                    native_support = masks['iris'] & ~masks['pupil']
                    if self.condition == 'E1-B2':
                        native_support &= ~masks['eyelashes']
                    support = preprocessing_transform_support(native_support, preprocessing_geometry(native_support.shape[1], native_support.shape[0]))
        image = preprocessing_tensor_from_pixels(pixels, self.condition, self.normalization, support=support, validity=validity, augmentation=self.augmentation)
        scalar = lambda x: torch.tensor(x, dtype=torch.long)
        return {'image': image, 'dataset_id': scalar(self.dataset_labels[row['dataset']]), 'subject_label': scalar(self.subject_labels[row['dataset'], row['subject_id']]), 'eye_label': scalar(0 if row['eye_side'] == 'L' else 1), 'iris_label': scalar(self.iris_labels[row['iris_id']]), 'source_label': scalar(self.source_labels[row['source_id']]), **{k: row[k] for k in ('image_id', 'iris_id', 'dataset', 'subject_id', 'eye_side', 'split')}}
data_FROZEN_MEMBERSHIP = {'seed': 42, 'datasets': {'CASIA-Interval': {'test': ['009', '010', '018', '023', '025', '028', '039', '044', '057', '070', '074', '096', '099', '101', '114', '125', '130', '131', '141', '149', '152', '153', '165', '175', '176', '182', '188', '191', '204', '205', '212', '218', '219', '231', '233', '234', '249'], 'train': ['001', '003', '004', '006', '007', '008', '011', '012', '013', '014', '015', '016', '019', '020', '021', '024', '026', '030', '032', '033', '034', '036', '037', '038', '040', '041', '042', '043', '045', '046', '047', '048', '051', '052', '053', '054', '055', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '071', '072', '073', '075', '076', '077', '078', '080', '081', '083', '084', '085', '086', '087', '089', '090', '092', '093', '095', '097', '102', '105', '106', '107', '108', '110', '111', '112', '113', '116', '117', '118', '119', '120', '121', '123', '124', '126', '127', '128', '129', '132', '133', '134', '135', '136', '137', '138', '139', '140', '142', '143', '144', '146', '147', '148', '150', '151', '154', '155', '156', '157', '158', '159', '160', '161', '163', '164', '166', '167', '168', '169', '170', '171', '172', '173', '178', '179', '181', '184', '185', '186', '187', '189', '190', '192', '193', '194', '195', '196', '197', '198', '200', '201', '202', '203', '206', '207', '209', '210', '211', '213', '214', '215', '216', '222', '223', '224', '225', '226', '227', '228', '229', '230', '232', '236', '237', '240', '241', '242', '243', '244', '245', '246', '248'], 'validation': ['002', '005', '017', '022', '027', '029', '031', '035', '049', '050', '056', '079', '082', '088', '091', '094', '098', '100', '103', '104', '109', '115', '122', '145', '162', '174', '177', '180', '183', '199', '208', '217', '220', '221', '235', '238', '239', '247']}, 'CUST-Iris': {'test': ['11', '17', '23', '28', '109', '114', '116', '120', '141', '142', '143', '147', '158', '173', '174', '176', '200', '216', '224', '226', '228', '244', '247', '248', '251', '252', '259', '264', '272', '279', '286', '295', '296', '302', '309', '313', '314', '320', '334', '355', '356', '359', '364', '365', '372', '376', '379', '382', '387', '393', '395', '413', '417', '423'], 'train': ['1', '2', '10', '12', '13', '14', '15', '16', '19', '21', '22', '24', '25', '27', '29', '30', '31', '32', '33', '34', '35', '37', '38', '39', '40', '41', '100', '102', '103', '104', '106', '108', '110', '111', '112', '113', '115', '118', '119', '121', '123', '124', '125', '126', '128', '129', '131', '132', '133', '134', '136', '137', '138', '139', '144', '145', '146', '148', '149', '150', '151', '154', '155', '156', '157', '159', '160', '161', '162', '163', '164', '165', '166', '167', '169', '170', '171', '172', '175', '177', '178', '179', '180', '181', '182', '183', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '201', '202', '203', '205', '206', '207', '209', '210', '211', '214', '215', '217', '218', '219', '220', '221', '222', '223', '225', '227', '229', '230', '232', '233', '234', '235', '236', '238', '239', '241', '242', '243', '245', '246', '250', '254', '255', '256', '257', '258', '260', '262', '263', '265', '266', '267', '269', '271', '273', '275', '276', '277', '278', '280', '281', '282', '283', '284', '285', '287', '288', '289', '290', '291', '292', '294', '297', '298', '299', '300', '301', '304', '307', '308', '310', '311', '312', '316', '317', '318', '319', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '336', '337', '339', '340', '341', '342', '343', '344', '345', '347', '348', '349', '352', '354', '360', '361', '362', '363', '366', '367', '368', '369', '370', '371', '373', '374', '375', '377', '378', '380', '381', '383', '386', '388', '389', '391', '392', '394', '397', '399', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412', '415', '416', '418', '419', '420', '421', '422', '424'], 'validation': ['3', '18', '20', '26', '36', '101', '105', '107', '117', '122', '127', '130', '135', '140', '152', '153', '168', '184', '185', '199', '204', '208', '212', '213', '231', '237', '240', '249', '253', '261', '268', '270', '274', '293', '303', '305', '306', '315', '321', '335', '338', '346', '350', '351', '353', '357', '358', '384', '385', '390', '396', '398', '400', '414']}, 'IITD': {'test': ['004', '013', '021', '026', '033', '034', '042', '059', '061', '064', '070', '072', '074', '081', '088', '092', '115', '117', '118', '128', '135', '153', '158', '166', '169', '174', '178', '189', '192', '198', '206', '214'], 'train': ['002', '003', '005', '007', '008', '009', '010', '011', '012', '014', '015', '016', '017', '018', '019', '020', '023', '024', '025', '027', '029', '030', '032', '035', '036', '037', '038', '039', '040', '041', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '055', '057', '060', '062', '063', '065', '066', '067', '068', '069', '076', '077', '078', '079', '080', '082', '083', '084', '085', '086', '091', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '105', '106', '107', '108', '109', '112', '113', '114', '116', '119', '120', '121', '122', '123', '126', '127', '129', '130', '131', '132', '133', '137', '138', '139', '140', '142', '143', '144', '145', '146', '147', '148', '149', '152', '154', '155', '156', '157', '159', '160', '162', '163', '164', '165', '167', '168', '170', '171', '172', '175', '176', '179', '181', '182', '183', '184', '185', '186', '187', '188', '190', '191', '195', '196', '197', '199', '200', '201', '202', '203', '204', '207', '208', '209', '210', '211', '212', '213', '215', '217'], 'validation': ['006', '022', '028', '031', '054', '056', '058', '071', '073', '075', '087', '089', '090', '093', '104', '110', '111', '124', '125', '134', '136', '141', '150', '151', '161', '173', '177', '180', '193', '194', '205', '216', '218']}}, 'excluded_image_ids': ['IITD:001_01_L', 'IITD:001_02_L', 'IITD:001_03_L', 'IITD:001_04_L', 'IITD:001_05_L', 'IITD:001_06_L', 'IITD:001_06_R', 'IITD:001_07_L', 'IITD:001_07_R', 'IITD:001_08_L', 'IITD:001_08_R', 'IITD:001_09_L', 'IITD:001_09_R', 'IITD:001_10_L', 'IITD:001_10_R', 'IITD:018_08_R', 'IITD:038_02_L', 'IITD:111_02_L', 'IITD:128_10_R', 'IITD:140_09_R', 'IITD:150_02_L', 'IITD:185_10_R', 'IITD:207_06_L', 'IITD:212_06_L', 'IITD:213_07_L'], 'excluded_subjects': [{'dataset': 'IITD', 'subject_id': '001'}]}
data_INVENTORY_COUNTS = {'raw': {'CASIA-Interval': {'train': 1847, 'validation': 396, 'test': 396}, 'CUST-Iris': {'train': 2016, 'validation': 432, 'test': 432}, 'IITD': {'train': 1509, 'validation': 327, 'test': 319}}, 'polar_common': {'CASIA-Interval': {'train': 1842, 'validation': 394, 'test': 395}, 'CUST-Iris': {'train': 2002, 'validation': 426, 'test': 427}, 'IITD': {'train': 1476, 'test': 308, 'validation': 318}}}

### ResNet-18 shared embedding encoder

In [ ]:
class resnet_IrisEmbeddingEncoder(nn.Module):

    def __init__(self, pretrained: bool=True, pretrained_file=None) -> None:
        super().__init__()
        self.backbone = timm.create_model('resnet18', pretrained=pretrained, num_classes=0, in_chans=1, global_pool='avg', **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        if self.backbone.num_features != 512:
            raise ValueError('Expected the 512-dimensional ResNet18 backbone')
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(512, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 320, 320):
            raise ValueError(f'Expected [B,1,320,320], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

### SwinV2-Tiny shared embedding encoder

In [ ]:
swin_MODEL_NAME = 'swinv2_tiny_window8_256'
swin_PRETRAINED_TAG = 'ms_in1k'
swin_ALLOWED_SHAPES = ((320, 320), (64, 512))

class swin_Float32WindowAttention(WindowAttention):
    """User-approved FP32 attention; preserve timm operations and parameter keys.

    Padded zero tokens produce nonfinite FP16 cosine-attention gradients in the
    installed timm implementation. Disable autocast for this module only. The
    patch embedding, MLPs, other backbone operations and head still use AMP.
    """

    def forward(self, x, mask=None):
        with torch.autocast(device_type=x.device.type, enabled=False):
            return super().forward(x.float(), None if mask is None else mask.float())

def swin_use_float32_attention(model):
    """Change execution only, without initializing tensors or consuming RNG."""
    modules = [module for module in model.modules() if type(module) is WindowAttention]
    if len(modules) != 12:
        raise ValueError(f'Expected 12 native SwinV2 attention modules, got {len(modules)}')
    for module in modules:
        module.__class__ = swin_Float32WindowAttention
    return len(modules)

class swin_SwinBackbone(nn.Module):
    """Expose the first patch convolution to the unchanged E1 gradient probe.

    The property does not register another module or duplicate any parameters.
    Windowing, pooling and pretrained loading stay in timm. Attention uses the
    approved float32 execution context around timm's unchanged calculation.
    """

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.model = timm.create_model(swin_MODEL_NAME, pretrained=pretrained, num_classes=0, in_chans=1, img_size=tuple(img_size), global_pool='avg', strict_img_size=False, **{'pretrained_cfg_overlay': {'file': str(pretrained_file), 'hf_hub_id': None}} if pretrained_file else {})
        self.float32_attention_modules = swin_use_float32_attention(self.model)
        self.num_features = int(self.model.num_features)
        if self.num_features != 768:
            raise ValueError(f'Unexpected SwinV2-Tiny features: {self.num_features}')
        if self.model.pretrained_cfg['tag'] != swin_PRETRAINED_TAG:
            raise ValueError('Unexpected pretrained tag')

    @property
    def conv1(self):
        return self.model.patch_embed.proj

    def forward(self, images):
        features = self.model(images)
        if features.ndim != 2 or features.shape != (images.shape[0], self.num_features):
            raise ValueError(f'Expected pooled [B,{self.num_features}], got {tuple(features.shape)}')
        return features

class swin_SwinEmbeddingEncoder(nn.Module):

    def __init__(self, img_size, pretrained=True, pretrained_file=None):
        super().__init__()
        self.img_size = tuple(img_size)
        if self.img_size not in swin_ALLOWED_SHAPES:
            raise ValueError(f'Unapproved E1 input shape: {self.img_size}')
        self.backbone = swin_SwinBackbone(self.img_size, pretrained=pretrained, pretrained_file=pretrained_file)
        self.backbone.requires_grad_(True)
        self.projection = nn.Sequential(nn.Linear(self.backbone.num_features, 512), nn.GELU(), nn.LayerNorm(512), nn.Linear(512, 256))

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, *self.img_size):
            raise ValueError(f'Expected [B,1,{self.img_size[0]},{self.img_size[1]}], got {tuple(images.shape)}')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def swin_describe_windows(encoder):
    return [dict(stage=i, input_resolution=list(stage.input_resolution), output_resolution=list(stage.output_resolution), windows=[list(block.window_size) for block in stage.blocks], shifts=[list(block.shift_size) for block in stage.blocks], dynamic_masks=[bool(block.dynamic_mask) for block in stage.blocks]) for i, stage in enumerate(encoder.backbone.model.layers)]

### Model construction and strict input guards

In [ ]:
class models_PolarEncoder(resnet_IrisEmbeddingEncoder):
    """Original ResNet polar adapter: only the spatial guard differs."""

    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (1, 64, 512):
            raise ValueError('Expected [B,1,64,512] polar input')
        projected = self.projection(self.backbone(images))
        with torch.autocast(device_type=images.device.type, enabled=False):
            return F.normalize(projected.float(), p=2, dim=1)

def models_build_model(config, *, pretrained=True, pretrained_asset=None, allow_download=False):
    """Initialize independently, using the original timm RGB-to-gray adaptation.

    `pretrained=False` is reserved for clearly labelled software tests or loading
    a complete trained checkpoint, never a fallback after failed asset loading.
    """
    architecture, condition = (config['architecture_label'], config['condition'])
    if pretrained_asset is not None:
        pretrained_asset = Path(pretrained_asset).expanduser().resolve()
        if not pretrained_asset.is_file():
            raise FileNotFoundError(f'Missing pretrained asset: {pretrained_asset}. Download the documented original timm model.safetensors and set PRETRAINED_ASSET.')
    if pretrained and pretrained_asset is None and (not allow_download):
        raise FileNotFoundError('Provide PRETRAINED_ASSET or explicitly set ALLOW_PRETRAINED_DOWNLOAD=True. No random-weight fallback is used.')
    try:
        if architecture == 'ResNet-18':
            cls = models_PolarEncoder if condition.startswith('E1-C') else resnet_IrisEmbeddingEncoder
            model = cls(pretrained=pretrained, pretrained_file=pretrained_asset)
        elif architecture == 'SwinV2-Tiny':
            model = swin_SwinEmbeddingEncoder(config['input_shape'][1:], pretrained=pretrained, pretrained_file=pretrained_asset)
        else:
            raise ValueError(f'Unsupported architecture: {architecture}')
    except (OSError, RuntimeError) as error:
        raise RuntimeError(f'Could not initialize the original {architecture} pretrained source. Check the documented tagged asset and installed timm version. No alternative weights were substituted. {error}') from error
    if not all((p.requires_grad for p in model.parameters())):
        raise RuntimeError('All backbone and projection parameters must remain trainable')
    return model

def models_model_summary(model, config):
    return {'architecture': config['architecture_label'], 'condition': config['condition'], 'input_shape': ['B', *config['input_shape']], 'output_shape': ['B', 256], 'pooled_features': model.backbone.num_features, 'trainable_parameters': sum((p.numel() for p in model.parameters() if p.requires_grad)), 'single_shared_encoder': True, 'l2_normalized_output': True, 'swin_float32_window_attention': config['architecture_label'] == 'SwinV2-Tiny'}

### Complete ROC, EER and low-FAR metric definitions

In [ ]:
metrics_DOMAINS = ('CUST-Iris', 'CASIA-Interval', 'IITD')
metrics_EER_METHOD = 'Linear interpolation on the complete ROC: locate adjacent points where d=FNR-FPR crosses zero; alpha=d0/(d0-d1), EER=FPR0+alpha*(FPR1-FPR0). Tied scores move together. No fixed operating threshold is selected.'
metrics_TAR_METHOD = 'ROC-interpolated TAR on the complete ROC: at an exact repeated FPR select its maximum TPR; between knots interpolate from the last lower-FPR point to the first higher-FPR point, preserving future vertical jumps. These values are not empirical results at a fixed threshold.'

def metrics_scores(values, name: str) -> np.ndarray:
    result = np.asarray(values, dtype=np.float64)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError(f'{name} must be a nonempty finite one-dimensional score array')
    return result

def metrics_interpolated_eer(fpr: np.ndarray, tpr: np.ndarray) -> float:
    difference = 1.0 - tpr - fpr
    right = int(np.flatnonzero(difference <= 0)[0])
    if difference[right] == 0 or right == 0:
        return float(fpr[right])
    left = right - 1
    alpha = difference[left] / (difference[left] - difference[right])
    return float(fpr[left] + alpha * (fpr[right] - fpr[left]))

def metrics_interpolated_tar(fpr: np.ndarray, tpr: np.ndarray, far: float) -> float:
    right = int(np.searchsorted(fpr, far, side='right'))
    left = right - 1
    if right == len(fpr):
        return float(tpr[-1])
    if left < 0:
        return float(tpr[0])
    if fpr[left] == far:
        return float(tpr[left])
    alpha = (far - fpr[left]) / (fpr[right] - fpr[left])
    return float(tpr[left] + alpha * (tpr[right] - tpr[left]))

def metrics_verification_metrics(genuine_scores, impostor_scores) -> dict:
    """Return float64 metrics from scores for both classes, without tuning.

    AUC is the trapezoidal area of the full ROC. EER is obtained by interpolating
    the FNR-FPR zero crossing. FAR resolution is one false acceptance divided by
    the number of impostor comparisons; interpolation does not improve that
    empirical resolution or establish independent-trial confidence intervals.
    """
    genuine = metrics_scores(genuine_scores, 'Genuine scores')
    impostor = metrics_scores(impostor_scores, 'Impostor scores')
    labels = np.concatenate((np.ones(len(genuine), dtype=np.uint8), np.zeros(len(impostor), dtype=np.uint8)))
    scores = np.concatenate((genuine, impostor))
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1, drop_intermediate=False)
    resolution = 1.0 / len(impostor)
    notes = []
    for far in (0.01, 0.001):
        if far < resolution:
            notes.append(f'FAR={far:g} is below one observed false acceptance ({resolution:.12g}); its TAR is an ROC interpolation, not an empirically resolved fixed-threshold operating point.')
    return {'roc_auc': float(auc(fpr, tpr)), 'eer': metrics_interpolated_eer(fpr, tpr), 'tar_at_far_1pct': metrics_interpolated_tar(fpr, tpr, 0.01), 'tar_at_far_0_1pct': metrics_interpolated_tar(fpr, tpr, 0.001), 'num_genuine': int(len(genuine)), 'num_impostor': int(len(impostor)), 'far_resolution': resolution, 'eer_method': metrics_EER_METHOD, 'tar_method': metrics_TAR_METHOD, 'tar_is_roc_interpolated': True, 'low_far_notes': notes}

def metrics_numpy_embedding(value) -> np.ndarray:
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    result = np.asarray(value, dtype=np.float32)
    if result.ndim != 1 or not len(result) or (not np.isfinite(result).all()):
        raise ValueError('Each embedding must be a finite nonempty vector')
    return result

def metrics_pair_chunks(pairs, chunk_size: int):
    if hasattr(pairs, 'to_batches'):
        for batch in pairs.to_batches(max_chunksize=chunk_size):
            yield batch.to_pylist()
    elif isinstance(pairs, Sequence) and (not isinstance(pairs, (str, bytes))):
        for offset in range(0, len(pairs), chunk_size):
            yield pairs[offset:offset + chunk_size]
    else:
        raise TypeError('Pairs must be a sequence of dictionaries or a PyArrow Table')

def metrics_evaluate_validation(embeddings_by_id: Mapping, pairs_by_dataset: Mapping, chunk_size: int=65536) -> dict:
    """Score every frozen pair and macro-average the three validation EERs.

    No images are opened here. Embeddings are CPU float32 vectors, converted
    from tensors if needed and normalized before cosine dot products. Scoring
    is chunked without choosing, discarding or sampling pairs based on scores.
    Pair membership/subject/source eligibility is sealed by the input protocol;
    this function additionally rejects self-comparisons and domain mismatches.
    """
    if not isinstance(chunk_size, Integral) or chunk_size <= 0:
        raise ValueError('chunk_size must be a positive integer')
    if set(pairs_by_dataset) != set(metrics_DOMAINS):
        raise ValueError('Macro-validation requires exactly CUST-Iris, CASIA-Interval and IITD')
    if not embeddings_by_id:
        raise ValueError('Validation embeddings cannot be empty')
    ids = sorted(embeddings_by_id)
    vectors = [metrics_numpy_embedding(embeddings_by_id[key]) for key in ids]
    if len({v.shape for v in vectors}) != 1:
        raise ValueError('Validation embeddings must share one feature dimension')
    matrix = np.stack(vectors)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if not np.isfinite(norms).all() or (norms <= 0).any():
        raise ValueError('Validation embeddings must have finite positive norms')
    matrix = matrix / norms
    index = {key: i for i, key in enumerate(ids)}
    per_dataset = {}
    for domain in metrics_DOMAINS:
        genuine_parts, impostor_parts = ([], [])
        for rows in metrics_pair_chunks(pairs_by_dataset[domain], int(chunk_size)):
            left, right, genuine = ([], [], [])
            for row in rows:
                if row['dataset'] != domain:
                    raise ValueError('Pair dataset differs from its frozen domain group')
                a, b = (row['image_id_a'], row['image_id_b'])
                if a == b:
                    raise ValueError('Self/identical-image comparisons are forbidden')
                if a not in index or b not in index:
                    raise ValueError(f'Frozen validation pair lacks an embedding: {a}, {b}')
                if not isinstance(row['is_genuine'], (bool, np.bool_)):
                    raise ValueError('Pair is_genuine must be an explicit boolean')
                left.append(index[a])
                right.append(index[b])
                genuine.append(row['is_genuine'])
            if not left:
                continue
            similarities = np.einsum('ij,ij->i', matrix[left], matrix[right], dtype=np.float32)
            similarities = np.clip(similarities, -1.0, 1.0)
            mask = np.asarray(genuine, dtype=bool)
            genuine_parts.append(similarities[mask])
            impostor_parts.append(similarities[~mask])
        genuine_scores = np.concatenate(genuine_parts) if genuine_parts else np.empty(0)
        impostor_scores = np.concatenate(impostor_parts) if impostor_parts else np.empty(0)
        per_dataset[domain] = metrics_verification_metrics(genuine_scores, impostor_scores)
    return {'per_dataset': per_dataset, 'macro_eer': float(np.mean([per_dataset[d]['eer'] for d in metrics_DOMAINS], dtype=np.float64)), 'score': 'cosine similarity of L2-normalized float32 embeddings', 'selection_metric': 'unweighted macro-average of three per-dataset validation EERs'}

### Physical-eye verification-pair checks

In [ ]:
pairs_KINDS = ('genuine', 'impostor', 'opposite_eye_diagnostic')

def pairs_validate_pairs(rows, records, *, domain, split, allow_availability_subset=False):
    by_id = {r['image_id']: r for r in records}
    pairs, unavailable, seen = ([], [], set())
    for raw in rows:
        row = dict(raw)
        a, b = (str(row['image_id_a']), str(row['image_id_b']))
        if a not in by_id or b not in by_id:
            if allow_availability_subset:
                unavailable.append((a, b))
                continue
            raise ValueError(f'Frozen pair lacks a member in its declared population: {a}, {b}')
        left, right = (by_id[a], by_id[b])
        if left['split'] != split or right['split'] != split or left['dataset'] != domain or (right['dataset'] != domain):
            raise ValueError('Pair crosses a dataset or frozen partition boundary')
        if a == b or left['source_id'] == right['source_id']:
            raise ValueError('Self/identical-source images may not count as verification pairs')
        kind = row.get('pair_type')
        if not kind:
            kind = 'genuine' if data_explicit_bool(row['is_genuine'], 'is_genuine') else 'impostor'
        if kind not in pairs_KINDS:
            raise ValueError(f'Unexpected pair category: {kind}')
        genuine = left['iris_id'] == right['iris_id']
        impostor = left['subject_id'] != right['subject_id']
        opposite = left['subject_id'] == right['subject_id'] and left['eye_side'] != right['eye_side']
        if not {'genuine': genuine, 'impostor': impostor, 'opposite_eye_diagnostic': opposite}[kind]:
            raise ValueError(f'Pair labels disagree with dataset+subject+eye: {a}, {b}')
        if 'is_genuine' in row and data_explicit_bool(row['is_genuine'], 'is_genuine') != genuine:
            raise ValueError("Pair's is_genuine contradicts physical-eye identity")
        if 'label' in row and int(row['label']) != int(genuine):
            raise ValueError("Pair's label contradicts physical-eye identity")
        if 'dataset' in row and row['dataset'] != domain:
            raise ValueError("Pair's dataset disagrees with its members")
        if 'split' in row and row['split'] != split:
            raise ValueError("Pair's split disagrees with its members")
        if 'primary_evaluation' in row and data_explicit_bool(row['primary_evaluation'], 'primary_evaluation') != (kind != 'opposite_eye_diagnostic'):
            raise ValueError("Pair's primary/diagnostic flag contradicts its category")
        for side, member in (('a', left), ('b', right)):
            expected_fields = {f'iris_id_{side}': member['iris_id'], f'eye_side_{side}': member['eye_side'], f'partition_key_{side}': f"{domain}:{member['subject_id']}"}
            for key, expected in expected_fields.items():
                if key in row and row[key] != expected:
                    raise ValueError(f"Frozen pair's {key} contradicts the authoritative image label")
        key = tuple(sorted((a, b)))
        if key in seen:
            raise ValueError('Repeated/reversed pair in a frozen population')
        seen.add(key)
        pairs.append({**row, 'image_id_a': a, 'image_id_b': b, 'dataset': domain, 'split': split, 'pair_type': kind, 'is_genuine': bool(genuine), 'primary_evaluation': kind != 'opposite_eye_diagnostic'})
    return (pairs, unavailable)

def pairs_load_validation_pairs(files, records, condition, *, original_validation_records=None):
    if set(files) != set(config_DOMAINS):
        raise ValueError('The three original validation datasets and fixed pair files are required')
    result = {}
    for domain, path in files.items():
        original = original_validation_records if original_validation_records is not None else records
        if condition.startswith('E1-C') and original_validation_records is None:
            raise ValueError('Supply the complete frozen validation inventory before taking its pure-V2 availability view')
        pairs, _ = pairs_validate_pairs(data_load_table(path).to_dict('records'), original, domain=domain, split='validation')
        allowed = {r['image_id'] for r in records}
        if condition.startswith('E1-C'):
            pairs = [p for p in pairs if p['image_id_a'] in allowed and p['image_id_b'] in allowed]
        elif any((p['image_id_a'] not in allowed or p['image_id_b'] not in allowed for p in pairs)):
            raise ValueError('Cartesian validation pair population differs from retained validation membership')
        if not any((p['is_genuine'] for p in pairs)) or not any((p['pair_type'] == 'impostor' for p in pairs)):
            raise ValueError(f'Insufficient frozen validation pairs for {domain}')
        result[domain] = pairs
    return result

### RNG defaults

In [ ]:
def runtime_seed_everything(seed: int=42, deterministic: bool=True) -> None:
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)

## 3. Direct asset loading and frozen-model inference

In [ ]:
"""Visible notebook asset binding, inference and analysis helpers (no project imports)."""

DATASETS = ('CUST-Iris','CASIA-Interval','IITD','AMF','UTIRIS_NIR')
ORIGINAL_DATASETS = DATASETS[:3]
EXTERNAL_DATASETS = DATASETS[3:]
CONDITIONS = ('E1-A','E1-B1','E1-B2','E1-C1','E1-C2')
ARCHITECTURES = ('ResNet-18','SwinV2-Tiny')
METRICS = ('eer','roc_auc','tar_at_far_1pct','tar_at_far_0_1pct')
KINDS = ('genuine','impostor','opposite_eye_diagnostic')
verification_metrics = metrics_verification_metrics
roc_curve = roc_curve   # sklearn's complete-ROC implementation imported above
transform_support = preprocessing_transform_support

def required_path(path, explanation):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. {explanation} No alternate asset or population is substituted.')
    return path

def read_json(path):
    return json.loads(required_path(path,'Configure the documented asset path.').read_text(encoding='utf-8'))

def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n',encoding='utf-8')

def save_table(frame,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if not isinstance(frame,pd.DataFrame):frame=pd.DataFrame(frame)
    frame.to_csv(path.with_suffix('.csv'),index=False)
    frame.to_parquet(path.with_suffix('.parquet'),index=False)
    return frame

def config():return copy.deepcopy(ANALYSIS_CONFIG)
def settings():return copy.deepcopy(STAGE_CONFIG)

def original_plan():
    source=read_json(CHECKPOINT_INDEX)
    jobs=source['jobs']
    assert len(jobs)==30
    assert {(j['architecture'],j['condition'],j['seed']) for j in jobs}=={
        (a,c,s) for a in ARCHITECTURES for c in CONDITIONS for s in (42,123,2025)}
    return source

def jobs_for(conditions=CONDITIONS):
    return [j for j in original_plan()['jobs'] if j['condition'] in conditions]

def read_image(path,unused=None):
    path=required_path(path,'Supply the unchanged uint8 image/mask cache or native source.')
    a=cv2.imdecode(np.fromfile(path,np.uint8),cv2.IMREAD_UNCHANGED)
    if a is None or a.ndim!=2 or a.dtype!=np.uint8:
        raise ValueError(f'Expected original uint8 grayscale: {path}')
    return a

def read_support(path,unused=None):
    a=read_image(path)
    assert a.shape==(320,320) and np.isin(a,[0,255]).all()
    return a>0

def decode_native(path):
    path=required_path(path,'Supply the original native image.')
    a=cv2.imdecode(np.fromfile(path,np.uint8),cv2.IMREAD_GRAYSCALE|cv2.IMREAD_IGNORE_ORIENTATION)
    if a is None or a.ndim!=2 or a.dtype!=np.uint8:raise ValueError('Cannot decode native grayscale')
    return a

def _record_path(value):
    return str(data_resolve_path(value,DATA_ROOT,PATH_REMAP)) if data_present(value) else None

@lru_cache(maxsize=1)
def evaluation_rows():
    table=pd.read_parquet(required_path(EVALUATION_MANIFEST,'Supply the evaluation-only metadata asset described in ASSETS.md.'))
    result=[]
    for r in table.to_dict('records'):
        for k in ('image_id','dataset','subject_id','iris_id','eye_side','source_id','split'):r[k]=str(r[k])
        assert r['dataset'] in DATASETS
        assert r['split']==('test' if r['dataset'] in ORIGINAL_DATASETS else 'external')
        assert r['iris_id']==f"{r['dataset']}:{r['subject_id']}:{r['eye_side']}" and r['eye_side'] in ('L','R')
        for key in ('source_path','cached_path','b1_support_path','b2_support_path','c1_path','c2_path',
                    'c1_validity_path','c2_validity_path','geometry_path','coordinates_path','eyelash_footprint_path'):
            r[key]=_record_path(r.get(key))
        masks=r['mask_paths']
        if isinstance(masks,str):masks=json.loads(masks)
        r['mask_paths']=json.dumps({k:_record_path(v) for k,v in masks.items()})
        flags=r.get('quality_flags',[])
        if not isinstance(flags,str):flags=json.dumps(flags.tolist() if hasattr(flags,'tolist') else list(flags))
        r['quality_flags']=flags
        if not isinstance(r['raw_geometry_json'],str):r['raw_geometry_json']=json.dumps(r['raw_geometry_json'])
        # Original tensor-loader schema: coordinates and support stay in the same frame.
        r.update(native_path=r['source_path'],raw_path=r['cached_path'],
                 b1_status=r.get('segmentation_status','ok'),b2_status=r.get('segmentation_status','ok'),
                 geometry_version='v2',v1_fallback_used=False,v3_used=False,
                 failure_reason=r.get('polar_failure_reason',''))
        result.append(r)
    assert len({r['image_id'] for r in result})==len(result)
    assert len({r['source_id'] for r in result})==len(result),'Supply the existing exact-duplicate exclusions'
    # Use frozen membership to enforce the train/test boundary without loading train images.
    frozen=read_json(SUBJECT_SPLITS)['datasets']
    for r in result:
        if r['dataset'] in ORIGINAL_DATASETS:
            assert r['subject_id'] in frozen[r['dataset']]['test']
            assert all(r['subject_id'] not in frozen[r['dataset']][s] for s in ('train','validation'))
    return result

def manifest(dataset):
    rows=[r for r in evaluation_rows() if r['dataset']==dataset]
    if dataset in ORIGINAL_DATASETS:return sorted(rows,key=lambda r:r['image_id'])
    if any('inference_order' not in r for r in rows):
        raise ValueError('External metadata must retain the original inference_order; filename sorting is not a replacement.')
    return sorted(rows,key=lambda r:int(r['inference_order']))

def domain_views(dataset):
    rows=manifest(dataset)
    split='test' if dataset in ORIGINAL_DATASETS else 'external'
    return {c:data_condition_view(rows,c,split)[0] for c in CONDITIONS}

def dataset_loader(records,condition,dataset):
    assert records and all(r['dataset']==dataset for r in records)
    cfg=config_frozen_config('ResNet-18',condition,42,'cuda')
    return data_IrisDataset(records,cfg,split='test' if dataset in ORIGINAL_DATASETS else 'external',augment=False)

def load_population(dataset,population='matched'):
    root=(MATCHED_POPULATION if population=='matched' else OCCLUSION_POPULATION)/dataset
    members=pd.read_parquet(required_path(root/'membership.parquet','Supply the published frozen population metadata.'))
    pairs={k:pd.read_parquet(required_path(root/f'{k}.parquet','Supply the published frozen verification pairs.')) for k in KINDS}
    available={r['image_id']:r for r in manifest(dataset)}
    records=[available[x] for x in members.image_id]
    for kind,p in pairs.items():
        assert p.pair_id.is_unique
        pairs_validate_pairs(p.to_dict('records'),records,domain=dataset,split='test' if dataset in ORIGINAL_DATASETS else 'external')
    if population=='matched':
        views=domain_views(dataset)
        actual=set.intersection(*(set(r['image_id'] for r in views[c]) for c in CONDITIONS))
        assert actual==set(members.image_id),'Do not redefine frozen common eligibility'
    return members,pairs

def load_common(dataset):return load_population(dataset,'matched')

def load_model(job):
    assert torch.cuda.is_available(),'Original CUDA policy; no implicit CPU/precision fallback'
    cfg=config_frozen_config(job['architecture'],job['condition'],job['seed'],'cuda')
    runtime_seed_everything(job['seed'],cfg['runtime']['deterministic_algorithms'])
    path=required_path(CHECKPOINT_ROOT/job['checkpoint'],'Supply this exact validation-selected best checkpoint.')
    state=torch.load(path,map_location='cpu',weights_only=False)
    assert state['epoch']==state['best_epoch']==job['selected_epoch']
    assert state['config']['seed']==job['seed']
    assert state['config']['architecture']['name']==cfg['architecture']['name']
    assert state['config']['data'].get('representation','raw_ocular')==cfg['data']['representation']
    model=models_build_model(cfg,pretrained=False)
    model.load_state_dict(state['model_state'],strict=True)
    return model.eval().requires_grad_(False).cuda()

def infer(model,dataset,job):
    assert not model.training and all(not p.requires_grad for p in model.parameters())
    loader=DataLoader(dataset,batch_size=24,shuffle=False,num_workers=0)
    ids=[];vectors=[];torch.cuda.reset_peak_memory_stats();started=time.perf_counter()
    with torch.inference_mode():
        for batch in loader:
            assert set(batch['split'])<= {'test','external'}
            with torch.autocast('cuda',dtype=torch.float16,enabled=job['config']['runtime']['amp']):
                z=model(batch['image'].cuda())
            assert z.shape==(len(batch['image_id']),256) and torch.isfinite(z).all()
            assert torch.allclose(z.norm(dim=1),torch.ones(len(z),device='cuda'),atol=1e-5)
            ids.extend(batch['image_id']);vectors.append(z.float().cpu().numpy())
    assert len(ids)==len(set(ids))
    return ids,np.concatenate(vectors),dict(seconds=time.perf_counter()-started,
        cuda_peak_allocated_bytes=torch.cuda.max_memory_allocated(),batch_size=24,
        amp=job['config']['runtime']['amp'],normalization=config()['normalization'],training=False)

def release(model):
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def score_embeddings(ids,matrix,pairs):
    return {k:pair_scores(ids,matrix,p.to_dict('records')) for k,p in pairs.items()}

def full_inference_records(job,dataset):
    # Original in-domain inference batches spanned all three test datasets.
    if dataset in ORIGINAL_DATASETS:
        rows=sorted([r for r in evaluation_rows() if r['dataset'] in ORIGINAL_DATASETS],key=lambda r:r['image_id'])
        return data_condition_view(rows,job['condition'],'test')[0]
    return domain_views(dataset)[job['condition']]

def baseline_scores(dataset,job,pairs):
    # Reuse full original embedding caches, then score only frozen matched pairs.
    group='internal' if dataset in ORIGINAL_DATASETS else dataset
    path=EMBEDDING_ROOT/group/f"{job['name']}.npz"
    records=full_inference_records(job,dataset)
    if BASELINE_SOURCE=='saved_embeddings':
        required_path(path,'Supply the unchanged original full-population E1 embedding cache, or explicitly choose recompute.')
        with np.load(path,allow_pickle=False) as z:
            ids=z['image_ids'].astype(str).tolist();matrix=z['embeddings'].copy()
        assert ids==[r['image_id'] for r in records], 'Original inference membership AND order must be retained'
    elif BASELINE_SOURCE=='recompute':
        model=load_model(job)
        cfg=config_frozen_config(job['architecture'],job['condition'],job['seed'],'cuda')
        data=data_IrisDataset(records,cfg,split='test' if dataset in ORIGINAL_DATASETS else 'external',augment=False)
        ids,matrix,_=infer(model,data,job);del model;release(None)
    else:raise ValueError('Choose saved_embeddings or explicitly recompute; there is no fallback')
    assert matrix.dtype==np.float32 and matrix.shape==(len(ids),256)
    assert np.isfinite(matrix).all()
    np.testing.assert_allclose(np.linalg.norm(matrix,axis=1),1,atol=1e-5,rtol=0)
    return score_embeddings(ids,matrix,pairs)

def intervention_scores(stage,dataset,job,variant,pairs,make_data):
    # These are optional external assets from the completed study, not another notebook's outputs.
    path=INTERVENTION_EMBEDDING_ROOT/stage/dataset/job['name']/f'{variant}.npz'
    data=make_data()
    if INTERVENTION_SOURCE=='saved_embeddings':
        required_path(path,'Supply the original intervention embeddings or explicitly choose recompute.')
        with np.load(path,allow_pickle=False) as z:ids=z['image_ids'].astype(str).tolist();matrix=z['embeddings'].copy()
        assert ids==[r['image_id'] for r in data.records]
    elif INTERVENTION_SOURCE=='recompute':
        model=load_model(job);ids,matrix,_=infer(model,data,job);del model;release(None)
    else:raise ValueError('Unknown intervention source; no fallback')
    return score_embeddings(ids,matrix,pairs)

def metric_artifacts(scores,target,details):
    target=Path(target);target.mkdir(parents=True,exist_ok=False)
    result=verification_metrics(scores['genuine'],scores['impostor'])
    fpr,tpr,_=roc_curve(np.r_[np.ones(len(scores['genuine'])),np.zeros(len(scores['impostor']))],
        np.r_[scores['genuine'],scores['impostor']],drop_intermediate=False)
    np.savez_compressed(target/'scores.npz',**scores)
    np.savez_compressed(target/'roc.npz',fpr=fpr,tpr=tpr)
    record=dict(details,metrics=result,score_distributions={k:distribution(v) for k,v in scores.items()})
    write_json(target/'metrics.json',record)
    return record

def job_details(job,dataset,**extra):
    return dict(dataset=dataset,architecture=job['architecture'],condition=job['condition'],seed=job['seed'],
                selected_epoch=job['selected_epoch'],**extra)

def outcomes_with_baseline(members,pairs,scores,baseline,**keys):
    out=per_image_scores(members,pairs,scores)
    original=per_image_scores(members,pairs,baseline).set_index('image_id')
    for m in ('genuine_mean','impostor_p99','impostor_p999','difficulty_margin'):
        out[m+'_delta_baseline']=out[m]-out.image_id.map(original[m])
    return out.assign(**keys)

def save_sensitivity(images,metrics,loo,*,variant_column='variant'):
    images=save_table(pd.concat(images,ignore_index=True),OUTPUT_ROOT/'per_image_sensitivity')
    groups=['dataset','architecture','condition','seed',variant_column]
    groups=[g for g in groups if g in images]
    dc=[c for c in images if c.endswith('_delta_baseline')]
    if dc:
        save_table(images.groupby(groups+['subject_id'])[dc].mean().reset_index(),OUTPUT_ROOT/'subject_sensitivity')
    leave=save_table(pd.DataFrame(loo),OUTPUT_ROOT/'leave_one_subject_out_metrics')
    if len(leave) and variant_column=='variant':
        baseline_label='A_A' if (leave.variant=='A_A').any() else 'baseline'
        base=leave[leave.variant==baseline_label].set_index(['dataset','architecture','condition','seed','removed_subject'])
        changes=[]
        for r in leave[leave.variant!=baseline_label].to_dict('records'):
            key=tuple(r[k] for k in ['dataset','architecture','condition','seed','removed_subject'])
            b=base.loc[key]
            changes.append({**{k:r[k] for k in ['dataset','architecture','condition','seed','removed_subject','variant']},
                            **{m:r[m]-b[m] for m in METRICS}})
        if changes:save_table(changes,OUTPUT_ROOT/'leave_one_subject_out_baseline_effects')
    return images,leave

def start_output():
    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError('Choose a fresh OUTPUT_ROOT; recorded results and previous reproductions are never overwritten.')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    torch.set_num_threads(2);cv2.setNumThreads(1)

### Exact cosine scoring, distribution and subject-sensitivity definitions

In [ ]:
def mdtable(frame):
    if len(frame) == 0:
        return '(No records.)'
    return '\n'.join(['| ' + ' | '.join(map(str, frame.columns)) + ' |', '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'] + ['| ' + ' | '.join((str(x) for x in row)) + ' |' for row in frame.itertuples(index=False, name=None)])

def summary(frame, groups, metrics=METRICS):
    rows = []
    for keys, group in frame.groupby(groups, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        r = dict(zip(groups, keys))
        assert set(group.seed) == {42, 123, 2025} and len(group) == 3
        for metric in metrics:
            r[metric + '_mean'] = float(group[metric].mean())
            r[metric + '_sample_sd'] = float(group[metric].std(ddof=1))
            for seed in (42, 123, 2025):
                r[f'{metric}_seed{seed}'] = float(group.loc[group.seed == seed, metric].iloc[0])
        rows.append(r)
    return pd.DataFrame(rows)

def formatted_metrics(frame, groups):
    s = summary(frame, groups)
    for m in METRICS:
        scale = 1 if m == 'roc_auc' else 100
        s[m] = [f'{scale * a:.4f} ± {scale * b:.4f}' for a, b in zip(s[m + '_mean'], s[m + '_sample_sd'])]
    return s[list(groups) + list(METRICS)]

def filter_pairs(pairs, ids):
    return {k: p.loc[p.image_id_a.isin(ids) & p.image_id_b.isin(ids)].reset_index(drop=True) for k, p in pairs.items()}

def distribution(values):
    return dict(n=len(values), mean=float(np.mean(values)), sample_sd=float(np.std(values, ddof=1)), **dict(zip(('p01', 'p10', 'p50', 'p90', 'p99', 'p999', 'max'), map(float, np.quantile(values, [0.01, 0.1, 0.5, 0.9, 0.99, 0.999, 1])))))

def per_image_scores(members, pairs, scores):
    ids = members.image_id.tolist()
    lookup = {x: i for i, x in enumerate(ids)}
    gs = [[] for _ in ids]
    ns = [[] for _ in ids]
    for kind, lists in [('genuine', gs), ('impostor', ns)]:
        df = pairs[kind]
        v = scores[kind]
        for side in ('a', 'b'):
            ind = np.fromiter((lookup[x] for x in df['image_id_' + side]), int, count=len(df))
            order = np.argsort(ind, kind='stable')
            which = ind[order]
            starts = np.r_[0, np.flatnonzero(np.diff(which)) + 1, len(which)]
            for a, b in zip(starts[:-1], starts[1:]):
                lists[which[a]].extend(v[order[a:b]])
    out = members[['dataset', 'subject_id', 'iris_id', 'image_id', 'eye_side']].copy()
    out['genuine_count'] = [len(x) for x in gs]
    out['genuine_mean'] = [np.mean(x) if len(x) else np.nan for x in gs]
    out['impostor_count'] = [len(x) for x in ns]
    out['impostor_p99'] = [np.quantile(x, 0.99) if len(x) else np.nan for x in ns]
    out['impostor_p999'] = [np.quantile(x, 0.999) if len(x) else np.nan for x in ns]
    out['difficulty_margin'] = out.genuine_mean - out.impostor_p99
    return out

def loo_metrics(dataset, job, members, pairs, scores, variant):
    by = members.set_index('image_id').subject_id.to_dict()
    endpoint = {k: (p.image_id_a.map(by).to_numpy(), p.image_id_b.map(by).to_numpy()) for k, p in pairs.items()}

    def one(subject):
        subset = {k: scores[k][(a != subject) & (b != subject)] for k, (a, b) in endpoint.items() if k != 'opposite_eye_diagnostic'}
        m = verification_metrics(subset['genuine'], subset['impostor'])
        return dict(dataset=dataset, architecture=job['architecture'], condition=job['condition'], seed=job['seed'], removed_subject=subject, variant=variant, **{k: m[k] for k in (*METRICS, 'num_genuine', 'num_impostor')})
    with ThreadPoolExecutor(max_workers=3) as pool:
        return list(pool.map(one, sorted(members.subject_id.unique())))
def pair_scores(image_ids, embeddings, pairs, chunk_size=65536):
    """Same float32 cosine calculation as the original, with bounded scratch."""
    if chunk_size < 1 or len(set(image_ids)) != len(image_ids):
        raise ValueError('Positive chunk size and unique embedding IDs required')
    matrix = np.asarray(embeddings, dtype=np.float32).copy()
    if matrix.ndim != 2 or len(matrix) != len(image_ids) or (not np.isfinite(matrix).all()):
        raise ValueError('Invalid embedding matrix')
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if (norms <= 0).any():
        raise ValueError('Zero embedding')
    matrix /= norms
    index = {key: i for i, key in enumerate(image_ids)}
    result = np.empty(len(pairs), dtype=np.float32)
    for start in range(0, len(pairs), chunk_size):
        batch = pairs[start:start + chunk_size]
        a = [index[p['image_id_a']] for p in batch]
        b = [index[p['image_id_b']] for p in batch]
        result[start:start + len(batch)] = np.clip(np.einsum('ij,ij->i', matrix[a], matrix[b], dtype=np.float32), -1.0, 1.0)
    return result

## 3. Binary-only inputs and original native-mask audit

In [ ]:
VARIANTS=('segmented_grayscale','binary_geometry')
OUTCOMES=('genuine_mean','impostor_p99','impostor_p999','difficulty_margin')

def jobs():
    selected = jobs_for(('E1-B1', 'E1-B2'))
    expected = {(a, c, s) for a in ARCHITECTURES for c in ('E1-B1', 'E1-B2') for s in (42, 123, 2025)}
    assert len(selected) == 12
    assert {(j['architecture'], j['condition'], j['seed']) for j in selected} == expected
    return selected

def binary_tensor(mask):
    if mask.shape != (320, 320) or mask.dtype != np.bool_:
        raise ValueError('Binary input requires a Boolean 320x320 support; no repair or resize')
    n = settings()['normalization']
    # Same uint8 -> float32 / 255 -> frozen normalization as E1, with no texture access.
    pixels = mask.astype(np.uint8) * np.uint8(255)
    return (torch.from_numpy(pixels).unsqueeze(0).float().div(255) - n['mean']) / n['std']

class GeometryDataset(Dataset):
    """Loads support files only. Original grayscale images are never read here."""
    def __init__(self, rows, condition):
        if condition not in ('E1-B1', 'E1-B2'):
            raise ValueError('E2-G accepts only B1/B2 checkpoints')
        if not rows or len({r['image_id'] for r in rows}) != len(rows):
            raise ValueError('Expected nonempty unique image membership')
        if any(r['split'] not in ('test', 'external') for r in rows):
            raise ValueError('Training/validation images are outside E2-G')
        self.records = rows
        self.support = settings()['support'][condition]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        mask = read_support(Path(row[self.support + '_path']), row.get(self.support + '_sha256'))
        return dict(image=binary_tensor(mask), **{k: row[k] for k in
            ('dataset', 'subject_id', 'iris_id', 'eye_side', 'image_id', 'split')})

def aligned_inputs(dataset):
    members, pairs = load_common(dataset)
    rows_by_id = {r['image_id']: r for r in manifest(dataset)}
    rows = [rows_by_id[i] for i in members.image_id]
    assert len(rows) == len(set(members.image_id))
    for row in rows:
        assert row['dataset'] == dataset
        assert row['split'] == ('test' if dataset in ORIGINAL_DATASETS else 'external')
        assert row['iris_id'] == f"{dataset}:{row['subject_id']}:{row['eye_side']}"
    assert members.dataset.eq(dataset).all()
    return members, pairs, rows

def validate_pairs(members, pairs):
    by = members.set_index('image_id')
    seen = set()
    for kind, p in pairs.items():
        assert p.pair_id.is_unique and set(p.pair_id).isdisjoint(seen)
        seen.update(p.pair_id)
        a = by.loc[p.image_id_a].reset_index(drop=True)
        b = by.loc[p.image_id_b].reset_index(drop=True)
        assert (p.image_id_a != p.image_id_b).all()
        assert (a.source_id != b.source_id).all()
        assert (a.dataset == b.dataset).all()
        if kind == 'genuine':
            assert (a.iris_id == b.iris_id).all()
        elif kind == 'impostor':
            assert (a.subject_id != b.subject_id).all()
        else:
            assert kind == 'opposite_eye_diagnostic'
            assert (a.subject_id == b.subject_id).all() and (a.eye_side != b.eye_side).all()
    # Confirm genuine and primary impostors are exhaustive, not sampled anew.
    n = len(members)
    choose2 = lambda sizes: int(sum(s * (s - 1) // 2 for s in sizes))
    assert len(pairs['genuine']) == choose2(members.groupby('iris_id').size())
    assert len(pairs['impostor']) == n * (n - 1) // 2 - choose2(members.groupby('subject_id').size())


In [ ]:
def native_binary(path, unused_sha, shape):
    return preprocessing_read_binary(path, shape)

def audit_row(row):
    g = json.loads(row['raw_geometry_json'])
    p = json.loads(row['mask_paths'])
    h = json.loads(row.get('mask_sha256', '{}'))
    native = {k: native_binary(p[k], h.get(k), (row['height'], row['width']))
              for k in ('iris', 'pupil', 'eyelashes', 'annulus', 'visible_iris')}
    assert np.array_equal(native['annulus'], native['iris'] & ~native['pupil'])
    assert np.array_equal(native['visible_iris'], native['annulus'] & ~native['eyelashes'])
    b1 = read_support(Path(row['b1_support_path']), row.get('b1_support_sha256'))
    b2 = read_support(Path(row['b2_support_path']), row.get('b2_support_sha256'))
    assert np.array_equal(b1, transform_support(native['annulus'], g))
    assert np.array_equal(b2, transform_support(native['visible_iris'], g))
    assert not (b2 & ~b1).any()
    content = np.zeros((320, 320), bool)
    y, x = g['pad_top'], g['pad_left']
    content[y:y + g['content_height'], x:x + g['content_width']] = True
    assert not (b1 & ~content).any()
    pupil = transform_support(native['pupil'], g)
    assert not (b1 & pupil).any()
    r = {k: row[k] for k in ('dataset', 'subject_id', 'iris_id', 'eye_side', 'image_id', 'split', 'source_id', 'quality_flags')}
    r.update(width=row['width'], height=row['height'], content_pixels=int(content.sum()),
        padding_pixels=int((~content).sum()), b1_pixels=int(b1.sum()), b2_pixels=int(b2.sum()),
        predicted_lash_fraction=float((b1 & ~b2).sum() / b1.sum()) if b1.any() else 0.,
        b1_native_exact=True, b2_native_exact=True, b2_subset_b1=True, padding_black=True, pupil_black=True,
        grayscale_samples_used_in_binary_input=0, b1_empty=not bool(b1.any()), b2_empty=not bool(b2.any()))
    for key, mask in [('b1', b1), ('b2', b2)]:
        r[key + '_support_path'] = row[key + '_support_path']
        r[key + '_support_sha256'] = row.get(key + '_support_sha256')
        r[key + '_pixel_sha256'] = hashlib.sha256(mask.tobytes()).hexdigest()
        t = binary_tensor(mask)
        assert t.shape == (1, 320, 320) and t.dtype == torch.float32 and torch.isfinite(t).all()
        assert len(t.unique()) <= 2
    return r

def previews(dataset, rows, audit, target):
    ordered = audit.sort_values(['predicted_lash_fraction', 'image_id'], kind='stable').reset_index(drop=True)
    selected = []
    used = set()
    for q in (0., .5, 1.):
        at = int(round(q * (len(ordered) - 1)))
        candidates = sorted(range(len(ordered)), key=lambda i: (abs(i - at), i))
        i = next(i for i in candidates if ordered.iloc[i].subject_id not in used)
        row = ordered.iloc[i]
        used.add(row.subject_id)
        selected.append(dict(dataset=dataset, image_id=row.image_id, subject_id=row.subject_id,
            requested_lash_quantile=q, predicted_lash_fraction=row.predicted_lash_fraction))
    by = {r['image_id']: r for r in rows}
    fig, axes = plt.subplots(3, 4, figsize=(10, 8), constrained_layout=True)
    titles = ['B1 original grayscale', 'B1 binary geometry', 'B2 original grayscale', 'B2 binary geometry']
    for i, item in enumerate(selected):
        row = by[item['image_id']]
        raw_image = read_image(row['cached_path'])
        b1 = read_support(Path(row['b1_support_path']), row.get('b1_support_sha256'))
        b2 = read_support(Path(row['b2_support_path']), row.get('b2_support_sha256'))
        for j, im in enumerate((raw_image * b1, b1.astype(np.uint8) * 255,
                                raw_image * b2, b2.astype(np.uint8) * 255)):
            axes[i, j].imshow(im, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
            axes[i, j].axis('off')
            axes[i, j].set_title(titles[j], fontsize=10)
        axes[i, 0].text(0, -.05, item['image_id'] + '\nPredicted lash fraction: ' +
            f"{100 * item['predicted_lash_fraction']:.1f}%", transform=axes[i, 0].transAxes, fontsize=8, va='top')
    fig.suptitle(dataset + ': unchanged coordinates; binary inputs contain no grayscale texture', fontsize=12)
    fig.savefig(target / 'input_previews.png', dpi=160)
    fig.savefig(target / 'input_previews.pdf')
    plt.close(fig)
    save_table(selected, target / 'preview_selection')


## 4. Exact aggregation and publication plots

In [ ]:
def differences(frame, variant_column='variant'):
    keys = ['dataset', 'architecture', 'condition', 'seed']
    baseline = frame[frame[variant_column] == 'segmented_grayscale'].set_index(keys)
    rows = []
    for r in frame[frame[variant_column] == 'binary_geometry'].to_dict('records'):
        b = baseline.loc[tuple(r[k] for k in keys)]
        rows.append({**{k: r[k] for k in keys}, **{m: float(r[m] - b[m]) for m in METRICS}})
    return pd.DataFrame(rows)


In [ ]:
def read_table(name):
    return pd.read_parquet(OUTPUT_ROOT / (name + '.parquet'))

def figures(dataset, metrics):
    root = OUTPUT_ROOT / dataset / 'figures'
    root.mkdir(exist_ok=True)
    colors = {42: '#245EA5', 123: '#D27A18', 2025: '#458350'}
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
    for (a, c), ax in zip(((a, c) for a in ARCHITECTURES for c in ('E1-B1', 'E1-B2')), axes.flat):
        group = metrics[(metrics.architecture == a) & (metrics.condition == c)]
        for seed, sub in group.groupby('seed'):
            by = sub.set_index('variant')
            ax.plot([0, 1], [100 * by.loc[v, 'eer'] for v in VARIANTS], 'o-', color=colors[seed], label=str(seed))
        ax.axhline(50, color='gray', linestyle=':', linewidth=1, label='50% reference')
        ax.set_xticks([0, 1], ['Grayscale baseline', 'Binary geometry'])
        ax.set_ylabel('EER (%)')
        ax.set_title(a + ' / ' + c)
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=8)
    fig.suptitle(dataset + ': same frozen checkpoints, identical images and exhaustive pairs')
    fig.savefig(root / 'eer_paired.png', dpi=180)
    fig.savefig(root / 'eer_paired.pdf')
    plt.close(fig)
    fig, axes = plt.subplots(4, 2, figsize=(12, 16), constrained_layout=True)
    for i, (a, c) in enumerate((a, c) for a in ARCHITECTURES for c in ('E1-B1', 'E1-B2')):
        for job in [j for j in jobs() if j['architecture'] == a and j['condition'] == c]:
            for variant in VARIANTS:
                directory = ((OUTPUT_ROOT / dataset / 'runs' / job['name'] / 'segmented_grayscale') if variant == 'segmented_grayscale'
                             else OUTPUT_ROOT / dataset / 'runs' / job['name'] / 'binary_geometry')
                style = '--' if variant == 'segmented_grayscale' else '-'
                with np.load(directory / 'roc.npz') as z:
                    axes[i, 0].plot(z['fpr'], z['tpr'], style, color=colors[job['seed']], linewidth=1,
                        label=f"{job['seed']} / {'gray' if variant == 'segmented_grayscale' else 'binary'}")
                with np.load(directory / 'scores.npz') as z:
                    for kind, alpha in [('genuine', 1.), ('impostor', .5)]:
                        hist, edges = np.histogram(z[kind], bins=np.linspace(-1, 1, 101), density=True)
                        axes[i, 1].plot((edges[:-1] + edges[1:]) / 2, hist, style,
                            color=colors[job['seed']], alpha=alpha, linewidth=1,
                            label=f"{job['seed']} / {'gray' if variant == 'segmented_grayscale' else 'binary'} / {kind}")
        axes[i, 0].plot([0, 1], [0, 1], ':', color='gray', linewidth=1)
        axes[i, 0].set(xlabel='FAR', ylabel='TAR', title=a + ' / ' + c + ': complete ROCs', xlim=(0, 1), ylim=(0, 1))
        axes[i, 1].set(xlabel='Cosine similarity', ylabel='Density', title=a + ' / ' + c + ': per-seed distributions')
        axes[i, 0].legend(fontsize=7)
        axes[i, 1].legend(fontsize=6, ncol=2)
    fig.suptitle(dataset + ': no pooling across seeds or datasets; dashed=gray, solid=binary')
    fig.savefig(root / 'roc_and_score_distributions.png', dpi=150)
    fig.savefig(root / 'roc_and_score_distributions.pdf')
    plt.close(fig)

def format_individual(frame):
    out = frame[['architecture', 'condition', 'seed', 'variant', *METRICS]].copy()
    for m in METRICS:
        out[m] = [f'{v * (1 if m == "roc_auc" else 100):.5f}' for v in out[m]]
    return out

def render_e2g_reports():
    metrics, effects = read_table('individual_metrics'), read_table('paired_baseline_effects')
    coverage, collisions = read_table('coverage'), read_table('identical_mask_collisions')
    dist_changes = read_table('distribution_changes')
    influence = read_table('subject_influence_summary')
    overview = []
    for keys, group in metrics.groupby(['dataset', 'architecture', 'condition'], sort=False):
        row = dict(zip(['dataset', 'architecture', 'condition'], keys))
        for v, name in [('segmented_grayscale', 'baseline'), ('binary_geometry', 'binary')]:
            sub = group[group.variant == v]
            row[name + '_eer_percent'] = f'{100 * sub.eer.mean():.3f} ± {100 * sub.eer.std(ddof=1):.3f}'
            row[name + '_auc'] = f'{sub.roc_auc.mean():.5f} ± {sub.roc_auc.std(ddof=1):.5f}'
        effect = effects[(effects.dataset == keys[0]) & (effects.architecture == keys[1]) & (effects.condition == keys[2])]
        row['paired_eer_change_pp'] = f'{100 * effect.eer.mean():+.3f} ± {100 * effect.eer.std(ddof=1):.3f}'
        overview.append(row)
    overview = save_table(overview, OUTPUT_ROOT / 'result_overview')
    n = settings()['normalization']
    normalization = dict(mean=n['mean'], std=n['std'], black_normalized=(0-n['mean'])/n['std'],
        white_normalized=(1-n['mean'])/n['std'])
    write_json(OUTPUT_ROOT / 'input_values.json', normalization)
    sections = [
        '# E2-G: segmentation-mask geometry in Cartesian iris verification models',
        'This independent frozen-model input intervention was added after E1 and E2-A to E2-F. '
        'It evaluates the twelve original E1-B1/B2 best checkpoints: two architectures, two training '
        'representations, three seeds. All 60 binary-input evaluations use exactly the existing E2-A '
        'matched populations and exhaustive pairs. Original grayscale results are reused without modification. '
        'No training, fine-tuning, checkpoint selection, normalization estimation or threshold fitting occurred.',
        '## Exact inputs',
        '**B1 → B1 binary annulus:** white where iris AND NOT pupil, black everywhere else. '
        '**B2 → B2 binary support:** white where annulus AND NOT predicted eyelashes, black elsewhere. '
        'These are filled support masks, not outlines. Every input uses the original 320×320 coordinates '
        'and black padding. No grayscale samples, segmentation probabilities, antialiasing, additional '
        'resizing, cropping, polar conversion or image enhancement enter binary inference. B1/B2 supports '
        'are never exchanged between trained conditions. Native formulas and original nearest-exact '
        'letterbox transforms were reconstructed for every image and matched the existing cached masks exactly.',
        f'Frozen normalization: mean {n["mean"]}, std {n["std"]}; binary black maps to '
        f'{normalization["black_normalized"]:.9f} and white to {normalization["white_normalized"]:.9f}. '
        'The original 256-D normalized embedding encoders and per-checkpoint AMP settings are unchanged, '
        'including Swin dynamic attention masks and float32 window attention. All models run in eval mode '
        'with gradients disabled; no optimizer is constructed.',
        '## Matched populations and coverage', mdtable(coverage),
        'CUST-Iris, CASIA-Interval and IITD contain only their original frozen test subjects; AMF and UTIRIS '
        'NIR remain independent external domains. There are 2,423 images total, evaluated independently '
        'within each dataset. No training/validation image is added. Same-person opposite eyes are a '
        'separate diagnostic and never primary impostors. Exact-source identical-image pairs are absent. '
        'No new subject assignments, exclusions, image eligibility or pair generation occurred.',
        '## Main results',
        'Values are mean ± sample SD across all three seeds (42, 123, 2025). EER is percent; AUC is [0,1]; '
        'paired changes are binary minus the SAME checkpoint’s grayscale baseline, in percentage points. '
        'No best seed is selected.', mdtable(overview),
        '## Identical-mask collisions', mdtable(collisions),
        'These are identical binary geometries from distinct, nonidentical source images. They remain '
        'in both genuine and impostor populations. High cosine similarity for such pairs is an expected '
        'consequence of discarding grayscale texture, not a reason for exclusion.',
        '## Interpretation and practical limits',
        'The intervention removes all grayscale iris texture. It retains image-relative boundary shape, '
        'iris/pupil position and size, visible-support fragmentation, predicted occlusion geometry, '
        'and the B2 predicted eyelash footprint. These include acquisition and segmentation effects, '
        'so the experiment is not a pure clean-anatomy identity test. Binary foreground 255 also changes '
        'input statistics and edge contrast substantially from training. Non-chance performance is '
        'descriptive evidence of discrimination available to a frozen model under this specific intervention; '
        'it does not establish that segmentation geometry caused its original grayscale accuracy. '
        'Degradation or chance-like performance does not prove geometry carries no identity information '
        'or that a model trained directly on masks would fail. Predicted masks are not anatomical ground truth.',
        'Chance references of EER 50% and AUC 0.5 are descriptive, without a significance claim. '
        'TAR@FAR uses the original complete-ROC interpolation and is not a measured fixed-threshold operating '
        'point. Repeated pairs and captures are dependent. Sample SD across three training seeds is not '
        'a participant confidence interval. Subject-removal analyses measure influence, not independent-trial '
        'uncertainty. Comparisons do not pool any dataset score distributions or replace prior E1/E2 results.'
    ]
    for d in DATASETS:
        sub = metrics[metrics.dataset == d]
        eff = effects[effects.dataset == d].assign(variant='binary-minus-grayscale')
        frames = ['# ' + d + ': E2-G', '## Absolute metrics',
            'EER/TAR are percent, AUC [0,1]; mean ± sample SD over the three seeds.',
            mdtable(formatted_metrics(sub, ['dataset', 'architecture', 'condition', 'variant'])),
            '## Paired within-checkpoint differences',
            'EER/TAR differences are percentage points; AUC differences are on [0,1].',
            mdtable(formatted_metrics(eff, ['dataset', 'architecture', 'condition', 'variant'])),
            '## All individual seeds', mdtable(format_individual(sub)),
            '## Similarity-distribution changes',
            'Values are binary minus grayscale, averaged within numerical seed, then mean/sample SD across seeds. '
            'Genuine mean and impostor p99/p99.9 distinguish score shifts from verification changes. '
            'Opposite-eye scores remain diagnostic.',
            mdtable(summary(dist_changes[dist_changes.dataset == d], ['dataset', 'architecture', 'condition', 'pair_kind'],
                metrics=('mean', 'p99', 'p999'))[['dataset', 'architecture', 'condition', 'pair_kind',
                    'mean_mean', 'mean_sample_sd', 'p99_mean', 'p99_sample_sd', 'p999_mean', 'p999_sample_sd']].round(6)),
            '## Subject influence', mdtable(influence[influence.dataset == d].round(5)),
            'Full per-image, subject-averaged and leave-one-subject-out values are in the corresponding '
            'CSV/Parquet tables. LOO removes all comparisons involving each omitted subject.',
            '## Previews and figures',
            'Biometric input previews are available only when explicitly generated locally.', '![Paired EER](figures/eer_paired.png)',
            '![Complete ROCs and distributions](figures/roc_and_score_distributions.png)']
        (OUTPUT_ROOT / d / 'E2_G_REPORT.md').write_text('\n\n'.join(frames) + '\n', encoding='utf-8')
        figures(d, sub)
        sections.extend(frames[:frames.index('## Previews and figures')])
        sections.append(f'[{d}: previews, figures and independent report]({d}/E2_G_REPORT.md)')
    sections += ['## Reproduction', 'The standalone E2-G notebook contains all input, model, scoring, analysis, report and plot definitions. Original best checkpoints and frozen populations remain unchanged. New outputs are stored separately.']
    (OUTPUT_ROOT / 'REPORT.md').write_text('\n\n'.join(sections) + '\n', encoding='utf-8')
    print('E2-G detailed report, all per-dataset reports and figures generated.', flush=True)


## 5. Complete experiment orchestration

In [ ]:
def run_e2_g():
    start_output()
    individual, distributions, images, leave = [], [], [], []
    audits, coverage, collisions = [], [], []
    populations = {}
    for dataset in DATASETS:
        members, pairs, rows = aligned_inputs(dataset)
        validate_pairs(members, pairs)
        target = OUTPUT_ROOT / dataset
        target.mkdir()
        with ThreadPoolExecutor(max_workers=3) as pool:
            audit = save_table(list(pool.map(audit_row, rows)), target / 'input_audit')
        audits.append(audit)
        coverage.append(dict(dataset=dataset, images=len(members), subjects=members.subject_id.nunique(),
            iris_ids=members.iris_id.nunique(), left=int(members.eye_side.eq('L').sum()), right=int(members.eye_side.eq('R').sum()),
            **{k:len(p) for k,p in pairs.items()}, b1_empty=int(audit.b1_empty.sum()), b2_empty=int(audit.b2_empty.sum()),
            all_native_masks_exact=bool(audit.b1_native_exact.all() and audit.b2_native_exact.all()),
            all_b2_subset_b1=bool(audit.b2_subset_b1.all())))
        for condition,key in [('E1-B1','b1'),('E1-B2','b2')]:
            by=audit.set_index('image_id')[key+'_pixel_sha256']
            for kind,p in pairs.items():
                same=p.image_id_a.map(by).to_numpy()==p.image_id_b.map(by).to_numpy()
                collisions.append(dict(dataset=dataset,condition=condition,pair_kind=kind,pairs=len(p),
                    identical_binary_masks=int(same.sum()),distinct_masks=int(by.nunique())))
        save_table(members,target/'membership')
        if SAVE_LOCAL_BIOMETRIC_PREVIEWS:
            previews(dataset,rows,audit,target)
        populations[dataset]=(members,pairs,rows)
    save_table(pd.concat(audits,ignore_index=True),OUTPUT_ROOT/'input_audit')
    save_table(coverage,OUTPUT_ROOT/'coverage')
    save_table(collisions,OUTPUT_ROOT/'identical_mask_collisions')
    for job in jobs():
        model=None
        for dataset in DATASETS:
            members,pairs,rows=populations[dataset]
            root=OUTPUT_ROOT/dataset/'runs'/job['name']
            root.mkdir(parents=True)
            base_scores=baseline_scores(dataset,job,pairs)
            data=GeometryDataset(rows,job['condition'])
            cache=INTERVENTION_EMBEDDING_ROOT/'E2-G'/dataset/job['name']/'binary_geometry.npz'
            if INTERVENTION_SOURCE=='saved_embeddings':
                required_path(cache,'Supply the completed E2-G binary embeddings, or explicitly choose recompute.')
                with np.load(cache,allow_pickle=False) as z:
                    ids=z['image_ids'].astype(str).tolist();matrix=z['embeddings'].copy()
                runtime={'saved_binary_embeddings':True,'training':False}
            elif INTERVENTION_SOURCE=='recompute':
                if model is None:
                    model=load_model(job)
                ids,matrix,runtime=infer(model,data,job)
            else:
                raise ValueError('Choose saved_embeddings or recompute explicitly; no fallback')
            assert ids==members.image_id.tolist(), 'Preserve the original E2-G matched-membership inference order'
            assert matrix.dtype==np.float32 and matrix.shape==(len(ids),256) and np.isfinite(matrix).all()
            np.testing.assert_allclose(np.linalg.norm(matrix,axis=1),1,atol=1e-5,rtol=0)
            geometry_scores=score_embeddings(ids,matrix,pairs)
            base_image=per_image_scores(members,pairs,base_scores).set_index('image_id')
            for variant,scores in [('segmented_grayscale',base_scores),('binary_geometry',geometry_scores)]:
                target=root/variant
                record=metric_artifacts(scores,target,job_details(job,dataset,variant=variant,
                    training=False,threshold_fitting=False,grayscale_samples_in_binary_input=0,
                    inference=runtime if variant=='binary_geometry' else {'reused_original_embeddings':True}))
                if variant=='binary_geometry':
                    np.savez_compressed(target/'embeddings.npz',image_ids=np.asarray(ids),embeddings=matrix)
                individual.append(dict(dataset=dataset,architecture=job['architecture'],condition=job['condition'],
                    seed=job['seed'],variant=variant,selected_epoch=job['selected_epoch'],
                    **{m:record['metrics'][m] for m in (*METRICS,'num_genuine','num_impostor','far_resolution')}))
                for kind,dist in record['score_distributions'].items():
                    distributions.append(dict(dataset=dataset,architecture=job['architecture'],condition=job['condition'],
                        seed=job['seed'],variant=variant,pair_kind=kind,**dist))
                image=per_image_scores(members,pairs,scores)
                for key in OUTCOMES:
                    image[key+'_delta_baseline']=image[key]-image.image_id.map(base_image[key])
                image=image.assign(architecture=job['architecture'],condition=job['condition'],seed=job['seed'],variant=variant)
                images.append(image)
                if variant=='binary_geometry':
                    save_table(image,target/'per_image_sensitivity')
                # Original B1/B2 baseline LOO diagnostics were derived directly
                # from saved baseline scores; no baseline model inference needed.
                leave.append(pd.DataFrame(loo_metrics(dataset,job,members,pairs,scores,variant)))
            print(dataset,job['name'],'completed')
        if model is not None:
            del model
            release(None)

    metrics = save_table(individual, OUTPUT_ROOT / 'individual_metrics')
    save_table(summary(metrics, ['dataset', 'architecture', 'condition', 'variant']), OUTPUT_ROOT / 'three_seed_summary')
    effects = save_table(differences(metrics), OUTPUT_ROOT / 'paired_baseline_effects')
    save_table(summary(effects, ['dataset', 'architecture', 'condition']), OUTPUT_ROOT / 'paired_effect_summary')
    dist = save_table(distributions, OUTPUT_ROOT / 'score_distributions')
    changes = []
    for keys, g in dist.groupby(['dataset', 'architecture', 'condition', 'seed', 'pair_kind']):
        by = g.set_index('variant')
        b, r = by.loc['segmented_grayscale'], by.loc['binary_geometry']
        changes.append(dict(zip(['dataset', 'architecture', 'condition', 'seed', 'pair_kind'], keys),
            **{k: float(r[k] - b[k]) for k in ('mean', 'p10', 'p50', 'p90', 'p99', 'p999', 'max')}))
    save_table(changes, OUTPUT_ROOT / 'distribution_changes')
    image = save_table(pd.concat(images, ignore_index=True), OUTPUT_ROOT / 'per_image_sensitivity')
    outcomes = [*OUTCOMES, *(m + '_delta_baseline' for m in OUTCOMES)]
    save_table(image.groupby(['dataset', 'architecture', 'condition', 'seed', 'variant', 'subject_id']).agg(
        images=('image_id', 'size'), **{k: (k, 'mean') for k in outcomes}).reset_index(), OUTPUT_ROOT / 'subject_sensitivity')
    loo = save_table(pd.concat(leave, ignore_index=True), OUTPUT_ROOT / 'leave_one_subject_out_metrics')
    keys = ['dataset', 'architecture', 'condition', 'seed', 'removed_subject']
    baseline = loo[loo.variant == 'segmented_grayscale'].set_index(keys)
    deltas = []
    for r in loo[loo.variant == 'binary_geometry'].to_dict('records'):
        b = baseline.loc[tuple(r[k] for k in keys)]
        deltas.append({**{k: r[k] for k in keys}, **{m: float(r[m] - b[m]) for m in METRICS}})
    influence = save_table(deltas, OUTPUT_ROOT / 'leave_one_subject_out_baseline_effects')
    rows = []
    for keys, group in influence.groupby(['dataset', 'architecture', 'condition']):
        full = effects[(effects.dataset == keys[0]) & (effects.architecture == keys[1]) & (effects.condition == keys[2])].eer.mean()
        removed = group.groupby('removed_subject').eer.mean()
        shift = removed - full
        rows.append(dict(zip(['dataset', 'architecture', 'condition'], keys), full_mean_eer_effect_pp=100 * full,
            loo_min_effect_pp=100 * removed.min(), loo_max_effect_pp=100 * removed.max(),
            sign_reversals=int((removed * full < 0).sum()), most_influential_subject=str(shift.abs().idxmax()),
            largest_absolute_shift_pp=100 * shift.abs().max()))
    save_table(rows, OUTPUT_ROOT / 'subject_influence_summary')
    contrasts = []
    for keys, group in metrics.groupby(['dataset', 'condition', 'seed', 'variant']):
        by = group.set_index('architecture')
        contrasts.append(dict(zip(['dataset', 'condition', 'seed', 'variant'], keys),
            contrast='Swin-minus-ResNet', **{m: float(by.loc['SwinV2-Tiny', m] - by.loc['ResNet-18', m]) for m in METRICS}))
    contrast = save_table(contrasts, OUTPUT_ROOT / 'architecture_contrasts')
    save_table(summary(contrast, ['dataset', 'condition', 'variant', 'contrast']), OUTPUT_ROOT / 'architecture_contrast_summary')
    assert len(metrics) == 120 and len(effects) == 60
    render_e2g_reports()
    return metrics


## 6. Environment and inexpensive checks

In [ ]:
display(pd.Series(config_environment_info(),name="Environment"))
support=np.zeros((320,320),dtype=bool)
support[80:240,80:240]=True
tensor=binary_tensor(support)
assert tensor.shape==(1,320,320) and tensor.dtype==torch.float32 and tensor.unique().numel()==2
expected=(torch.from_numpy(support.astype(np.uint8)*255)[None].float()/255-STAGE_CONFIG['normalization']['mean'])/STAGE_CONFIG['normalization']['std']
assert torch.equal(tensor,expected)
chance=verification_metrics(np.ones(5),np.ones(17))
assert chance['eer']==.5 and chance['roc_auc']==.5
assert len(jobs())==12
print('Synthetic binary/normalization/tied-score checks passed; no recognition inference.')


## 7. Reproduce explicitly

In [ ]:
if RUN_ANALYSIS:
    reproduced=run_e2_g()
    display(reproduced.head())
else:
    print('Analysis disabled. Configure licensed assets and enable RUN_ANALYSIS explicitly.')


## 8. Inspect completed results

In [ ]:
if SHOW_RECORDED_RESULTS:
    if not RECORDED_RESULTS.is_dir():
        raise FileNotFoundError('Set RECORDED_RESULTS, or disable SHOW_RECORDED_RESULTS for a copied notebook.')
    print('Recorded completed-study results; not recalculated in this session.')
    for name in ('coverage.csv','result_overview.csv','three_seed_summary.csv','paired_effect_summary.csv'):
        display(pd.read_csv(RECORDED_RESULTS/name))


## Interpretation

This frozen-model binary-input intervention changes image statistics and edge contrast. Geometry includes position/scale, iris/pupil boundaries, fragmentation and predicted occlusion footprints. Discrimination does not establish what caused original grayscale accuracy; degradation does not prove geometry has no identity information. B1/B2 are independently trained checkpoints, so their comparison does not isolate the effect of a lash stencil alone. See the detailed report for every seed, EER/AUC/TAR, score tails and subject influence. All dataset populations remain independent.